# Data Processing For Modelling

This notebook contains the preprocessing done to participant rank data, feature engineering of lyrics and mood features, handling tie moods, feature engineering of genre, and data preparation for modelling.

The final data from this notebook is used for modelling in SupervisedLDA Notebook, MLP Notebook and RandomForest Notebook.

The data are:
* X_train_roberta
* X_test_roberta

* X_train_distilroberta
* X_test_distilroberta

* X_train_distilroberta_mitch
* X_test_distilroberta_mitch

* y_train
* y_test

* target_train
* target_test

A point to note is that the preprocessing of categorical and numerical features for modelling is not done here before the split to prevent data leakage.

In [25]:
# Import Library
import numpy as np
import pandas as pd
import unicodedata
import difflib
import re

# Participant Data Loading
Here, the questionnaire of participants are loaded. Missing values are filled using forward fill and the average of participants within the same demographic group and location is computed

In [26]:
def forward_fill_missing_rows(df):
    """
    Forward fill rows where all ranking columns are missing, using the values from the previous row.
    """
    # Identify columns to check for missing values
    rank_columns = ["5 - Definitely Will", "4 - Highly Likely", "3 - Might", "2 - Unlikely", "1 - Definitely Won’t"]
    
    # Forward fill rows where all rank columns are NaN
    for i in range(len(df)):
        if df.loc[i, rank_columns].isna().all():
            if i > 0:  # Ensure that there is a previous row
                # Fill missing values in the identified row with the values from the previous row
                df.loc[i, rank_columns] = df.loc[i - 1, rank_columns]
    
    return df

def transform_data(file_path, var_name="Likelihood", rank_column_name="Rank"):
    '''
    This function reads the excel file of each participant questionnaire for different location and
    transforms it from the wide data format to a long data format
    '''
    # Read the data from the Excel file
    data = pd.read_excel(file_path)
    
    # Apply forward fill for rows with all missing ranks
    df = forward_fill_missing_rows(data)

    # Define the columns to be melted
    value_vars = ["5 - Definitely Will", "4 - Highly Likely", "3 - Might", "2 - Unlikely", "1 - Definitely Won’t"]

    # Melt the DataFrame to get a single 'Rank' column
    melted_df = df.melt(id_vars=["S/N", "track_name", "artist_name"], value_vars=value_vars, var_name=var_name, value_name=rank_column_name)

    # Drop rows with NaN values in the rank column
    melted_df = melted_df.dropna(subset=[rank_column_name])

    # Sort the DataFrame by the "S/N" column to ensure it's in numerical order
    melted_df = melted_df.sort_values(by="S/N").reset_index(drop=True)

    # Drop the "Likelihood" column
    melted_df = melted_df.drop(columns=[var_name])

    return melted_df


def get_ordinal_suffix(number):
    """Return the ordinal suffix for a given number."""
    if 10 <= number % 100 <= 20:
        suffix = 'th'
    else:
        suffix = {1: 'st', 2: 'nd', 3: 'rd'}.get(number % 10, 'th')
    return suffix

def process_participant_data(participant_number, basepath='/Users/mac/Documents/Music recommendation/21-25_Male'):
    locations = ["Bar", "Bedroom", "Gym", "Mall", "Library", "TrainBus", "Restaurant", "Office"]
    base_path = f'{basepath}/Participant {participant_number}'
    
    ordinal_suffix = get_ordinal_suffix(participant_number)
    participant_dfs = {}

    for location in locations:
        file_path = f'{base_path}/{location}_{participant_number}{ordinal_suffix} participant.xlsx'
        rank_column_name = f"ranking_{participant_number}{ordinal_suffix}_participant"
        # Create a dynamic name for the DataFrame
        df_name = f"{location.lower()}_{participant_number}{ordinal_suffix}_participant"
        try:
            participant_dfs[df_name] = transform_data(file_path, var_name="likelihood", rank_column_name=rank_column_name)
        except Exception as e:
            print(f"Error processing {file_path}: {e}")
    
    return participant_dfs

def get_duplicate_sn_values(df, column_name='S/N'):
    # Get the value counts of the specified column
    value_counts = df[column_name].value_counts()
    # Filter the value counts to find instances with a count greater than 1
    duplicate_values = value_counts[value_counts > 1].index.tolist()
    return duplicate_values

def merge_participant_data(*dataframes):
    """Merge multiple DataFrames on common columns: 'S/N', 'track_name', 'artist_name'."""
    # Start with the first DataFrame
    merged_df = dataframes[0]

    # Merge remaining DataFrames
    for df in dataframes[1:]:
        merged_df = pd.merge(merged_df, df, on=['S/N', 'track_name', 'artist_name'], how='outer')

    return merged_df

def add_average_rank(df, average_rank_column_name):
    """
    Add a column to the DataFrame for the average of all ranking columns with a custom name.
    Then drop the original ranking columns.

    Parameters:
    df (pd.DataFrame): The DataFrame to modify.
    average_rank_column_name (str): The name to give to the new average rank column.

    Returns:
    pd.DataFrame: The modified DataFrame with the new average rank column.
    """
    # Identify the ranking columns (assuming they are the columns starting with 'ranking_')
    ranking_columns = [col for col in df.columns if col.startswith('ranking_')]

    # Calculate the average rank, ignoring NaN values
    df[average_rank_column_name] = df[ranking_columns].mean(axis=1)
    
    # Drop the original ranking columns
    df = df.drop(columns=ranking_columns)

    return df

def merge_dataframes_on_sn(df_list, merge_on='S/N'):
    """
    Merge multiple DataFrames on a specified column, avoiding suffixes for overlapping columns.

    Parameters:
    df_list (list): A list of DataFrames to merge.
    merge_on (str): The column name to merge the DataFrames on (default is 'S/N').

    Returns:
    pd.DataFrame: The merged DataFrame without suffixes for overlapping columns.
    """
    if len(df_list) < 2:
        raise ValueError("At least two DataFrames must be provided for merging.")

    # Start with the first DataFrame in the list
    merged_df = df_list[0]

    # Sequentially merge the remaining DataFrames, handling overlapping columns
    for df in df_list[1:]:
        overlapping_columns = [col for col in df.columns if col in merged_df.columns and col != merge_on]
        df = df.drop(columns=overlapping_columns)
        merged_df = merged_df.merge(df, on=merge_on, how='outer')

    return merged_df

## Male Participant between age 21 - 25

In [27]:
basepath='/Users/mac/Documents/Music recommendation/21-25_Male'
participant_1_dfs = process_participant_data(1)
participant_2_dfs = process_participant_data(2)
participant_3_dfs = process_participant_data(3)
participant_4_dfs = process_participant_data(4)
participant_5_dfs = process_participant_data(5)
participant_6_dfs = process_participant_data(6)
participant_7_dfs = process_participant_data(7)
participant_8_dfs = process_participant_data(8)
participant_9_dfs = process_participant_data(9)
participant_10_dfs = process_participant_data(10)

# Bar Ranked Data for Male participant between 21 and 25
bar_21_25_Male = merge_participant_data(
    participant_1_dfs.get('bar_1st_participant'), participant_2_dfs.get('bar_2nd_participant'), 
    participant_3_dfs.get('bar_3rd_participant'), participant_4_dfs.get('bar_4th_participant'), 
    participant_5_dfs.get('bar_5th_participant'), participant_6_dfs.get('bar_6th_participant'),
    participant_7_dfs.get('bar_7th_participant'), participant_8_dfs.get('bar_8th_participant'), 
    participant_9_dfs.get('bar_9th_participant'), participant_10_dfs.get('bar_10th_participant')
)
# Compute average rank and drop the original ranking columns
bar_21_25_Male = add_average_rank(bar_21_25_Male, '21_25_Male_AR')

# Bedroom Ranked Data for 10 Male participants between 21 and 25
bedroom_21_25_Male = merge_participant_data(
    participant_1_dfs.get('bedroom_1st_participant'), participant_2_dfs.get('bedroom_2nd_participant'), 
    participant_3_dfs.get('bedroom_3rd_participant'), participant_4_dfs.get('bedroom_4th_participant'), 
    participant_5_dfs.get('bedroom_5th_participant'), participant_6_dfs.get('bedroom_6th_participant'),
    participant_7_dfs.get('bedroom_7th_participant'), participant_8_dfs.get('bedroom_8th_participant'), 
    participant_9_dfs.get('bedroom_9th_participant'), participant_10_dfs.get('bedroom_10th_participant')
)
# Compute average rank
bedroom_21_25_Male = add_average_rank(bedroom_21_25_Male, '21_25_Male_AR')

# Gym Ranked Data for 10 Male participants between 21 and 25
gym_21_25_Male = merge_participant_data(
    participant_1_dfs.get('gym_1st_participant'), participant_2_dfs.get('gym_2nd_participant'), 
    participant_3_dfs.get('gym_3rd_participant'), participant_4_dfs.get('gym_4th_participant'), 
    participant_5_dfs.get('gym_5th_participant'), participant_6_dfs.get('gym_6th_participant'),
    participant_7_dfs.get('gym_7th_participant'), participant_8_dfs.get('gym_8th_participant'), 
    participant_9_dfs.get('gym_9th_participant'), participant_10_dfs.get('gym_10th_participant')
)
# Compute average rank 
gym_21_25_Male = add_average_rank(gym_21_25_Male, '21_25_Male_AR')

# Mall Ranked Data for 10 Male participants between 21 and 25
mall_21_25_Male = merge_participant_data(
    participant_1_dfs.get('mall_1st_participant'), participant_2_dfs.get('mall_2nd_participant'), 
    participant_3_dfs.get('mall_3rd_participant'), participant_4_dfs.get('mall_4th_participant'), 
    participant_5_dfs.get('mall_5th_participant'), participant_6_dfs.get('mall_6th_participant'),
    participant_7_dfs.get('mall_7th_participant'), participant_8_dfs.get('mall_8th_participant'), 
    participant_9_dfs.get('mall_9th_participant'), participant_10_dfs.get('mall_10th_participant')
)
# Compute average rank 
mall_21_25_Male = add_average_rank(mall_21_25_Male, '21_25_Male_AR')

# Library Ranked Data for 10 Male participants between 21 and 25
library_21_25_Male = merge_participant_data(
    participant_1_dfs.get('library_1st_participant'), participant_2_dfs.get('library_2nd_participant'), 
    participant_3_dfs.get('library_3rd_participant'), participant_4_dfs.get('library_4th_participant'), 
    participant_5_dfs.get('library_5th_participant'), participant_6_dfs.get('library_6th_participant'),
    participant_7_dfs.get('library_7th_participant'), participant_8_dfs.get('library_8th_participant'), 
    participant_9_dfs.get('library_9th_participant'), participant_10_dfs.get('library_10th_participant')
)
# Compute average rank 
library_21_25_Male = add_average_rank(library_21_25_Male, '21_25_Male_AR')

# TrainBus Ranked Data for 10 Male participants between 21 and 25
trainbus_21_25_Male = merge_participant_data(
    participant_1_dfs.get('trainbus_1st_participant'), participant_2_dfs.get('trainbus_2nd_participant'), 
    participant_3_dfs.get('trainbus_3rd_participant'), participant_4_dfs.get('trainbus_4th_participant'), 
    participant_5_dfs.get('trainbus_5th_participant'), participant_6_dfs.get('trainbus_6th_participant'),
    participant_7_dfs.get('trainbus_7th_participant'), participant_8_dfs.get('trainbus_8th_participant'), 
    participant_9_dfs.get('trainbus_9th_participant'), participant_10_dfs.get('trainbus_10th_participant')
)
# Compute average rank
trainbus_21_25_Male = add_average_rank(trainbus_21_25_Male, '21_25_Male_AR')

# Restaurant Ranked Data for 10 Male participants between 21 and 25
restaurant_21_25_Male = merge_participant_data(
    participant_1_dfs.get('restaurant_1st_participant'), participant_2_dfs.get('restaurant_2nd_participant'), 
    participant_3_dfs.get('restaurant_3rd_participant'), participant_4_dfs.get('restaurant_4th_participant'), 
    participant_5_dfs.get('restaurant_5th_participant'), participant_6_dfs.get('restaurant_6th_participant'),
    participant_7_dfs.get('restaurant_7th_participant'), participant_8_dfs.get('restaurant_8th_participant'), 
    participant_9_dfs.get('restaurant_9th_participant'), participant_10_dfs.get('restaurant_10th_participant')
)
# Compute average rank 
restaurant_21_25_Male = add_average_rank(restaurant_21_25_Male, '21_25_Male_AR')

# Office Ranked Data for 10 Male participants between 21 and 25
office_21_25_Male = merge_participant_data(
    participant_1_dfs.get('office_1st_participant'), participant_2_dfs.get('office_2nd_participant'), 
    participant_3_dfs.get('office_3rd_participant'), participant_4_dfs.get('office_4th_participant'), 
    participant_5_dfs.get('office_5th_participant'), participant_6_dfs.get('office_6th_participant'),
    participant_7_dfs.get('office_7th_participant'), participant_8_dfs.get('office_8th_participant'), 
    participant_9_dfs.get('office_9th_participant'), participant_10_dfs.get('office_10th_participant')
)
# Compute average rank
office_21_25_Male = add_average_rank(office_21_25_Male, '21_25_Male_AR')

## Female Participant between age 21 - 25

In [28]:
basepath='/Users/mac/Documents/Music recommendation/21-25_Female'
participant_11_dfs = process_participant_data(11, basepath = basepath)
participant_12_dfs = process_participant_data(12, basepath = basepath)
participant_13_dfs = process_participant_data(13, basepath = basepath)
participant_14_dfs = process_participant_data(14, basepath = basepath)
participant_15_dfs = process_participant_data(15, basepath = basepath)
participant_16_dfs = process_participant_data(16, basepath = basepath)
participant_17_dfs = process_participant_data(17, basepath = basepath)
participant_18_dfs = process_participant_data(18, basepath = basepath)
participant_19_dfs = process_participant_data(19, basepath = basepath)
participant_20_dfs = process_participant_data(20, basepath = basepath)

# Bar Ranked Data for Male participant between 21 and 25
bar_21_25_Female = merge_participant_data(
    participant_11_dfs.get('bar_11th_participant'), participant_12_dfs.get('bar_12th_participant'), 
    participant_13_dfs.get('bar_13th_participant'), participant_14_dfs.get('bar_14th_participant'), 
    participant_15_dfs.get('bar_15th_participant'), participant_16_dfs.get('bar_16th_participant'),
    participant_17_dfs.get('bar_17th_participant'), participant_18_dfs.get('bar_18th_participant'), 
    participant_19_dfs.get('bar_19th_participant'), participant_20_dfs.get('bar_20th_participant')
)
# Compute average rank and drop the original ranking columns
bar_21_25_Female = add_average_rank(bar_21_25_Female, '21_25_Female_AR')

# Bedroom Ranked Data for 10 Male participants between 21 and 25
bedroom_21_25_Female = merge_participant_data(
    participant_11_dfs.get('bedroom_11th_participant'), participant_12_dfs.get('bedroom_12th_participant'), 
    participant_13_dfs.get('bedroom_13th_participant'), participant_14_dfs.get('bedroom_14th_participant'), 
    participant_15_dfs.get('bedroom_15th_participant'), participant_16_dfs.get('bedroom_16th_participant'),
    participant_17_dfs.get('bedroom_17th_participant'), participant_18_dfs.get('bedroom_18th_participant'), 
    participant_19_dfs.get('bedroom_19th_participant'), participant_20_dfs.get('bedroom_20th_participant')
)
# Compute average rank
bedroom_21_25_Female = add_average_rank(bedroom_21_25_Female, '21_25_Female_AR')

# Gym Ranked Data for 10 Male participants between 21 and 25
gym_21_25_Female = merge_participant_data(
    participant_11_dfs.get('gym_11th_participant'), participant_12_dfs.get('gym_12th_participant'), 
    participant_13_dfs.get('gym_13th_participant'), participant_14_dfs.get('gym_14th_participant'), 
    participant_15_dfs.get('gym_15th_participant'), participant_16_dfs.get('gym_16th_participant'),
    participant_17_dfs.get('gym_17th_participant'), participant_18_dfs.get('gym_18th_participant'), 
    participant_19_dfs.get('gym_19th_participant'), participant_20_dfs.get('gym_20th_participant')
)
# Compute average rank 
gym_21_25_Female = add_average_rank(gym_21_25_Female, '21_25_Female_AR')

# Mall Ranked Data for 10 Male participants between 21 and 25
mall_21_25_Female = merge_participant_data(
    participant_11_dfs.get('mall_11th_participant'), participant_12_dfs.get('mall_12th_participant'), 
    participant_13_dfs.get('mall_13th_participant'), participant_14_dfs.get('mall_14th_participant'), 
    participant_15_dfs.get('mall_15th_participant'), participant_16_dfs.get('mall_16th_participant'),
    participant_17_dfs.get('mall_17th_participant'), participant_18_dfs.get('mall_18th_participant'), 
    participant_19_dfs.get('mall_19th_participant'), participant_20_dfs.get('mall_20th_participant')
)
# Compute average rank 
mall_21_25_Female = add_average_rank(mall_21_25_Female, '21_25_Female_AR')

# Library Ranked Data for 10 Male participants between 21 and 25
library_21_25_Female = merge_participant_data(
    participant_11_dfs.get('library_11th_participant'), participant_12_dfs.get('library_12th_participant'), 
    participant_13_dfs.get('library_13th_participant'), participant_14_dfs.get('library_14th_participant'), 
    participant_15_dfs.get('library_15th_participant'), participant_16_dfs.get('library_16th_participant'),
    participant_17_dfs.get('library_17th_participant'), participant_18_dfs.get('library_18th_participant'), 
    participant_19_dfs.get('library_19th_participant'), participant_20_dfs.get('library_20th_participant')
)
# Compute average rank 
library_21_25_Female = add_average_rank(library_21_25_Female, '21_25_Female_AR')

# TrainBus Ranked Data for 10 Male participants between 21 and 25
trainbus_21_25_Female = merge_participant_data(
    participant_11_dfs.get('trainbus_11th_participant'), participant_12_dfs.get('trainbus_12th_participant'), 
    participant_13_dfs.get('trainbus_13th_participant'), participant_14_dfs.get('trainbus_14th_participant'), 
    participant_15_dfs.get('trainbus_15th_participant'), participant_16_dfs.get('trainbus_16th_participant'),
    participant_17_dfs.get('trainbus_17th_participant'), participant_18_dfs.get('trainbus_18th_participant'), 
    participant_19_dfs.get('trainbus_19th_participant'), participant_20_dfs.get('trainbus_20th_participant')
)
# Compute average rank
trainbus_21_25_Female = add_average_rank(trainbus_21_25_Female, '21_25_Female_AR')

# Restaurant Ranked Data for 10 Male participants between 21 and 25
restaurant_21_25_Female = merge_participant_data(
    participant_11_dfs.get('restaurant_11th_participant'), participant_12_dfs.get('restaurant_12th_participant'), 
    participant_13_dfs.get('restaurant_13th_participant'), participant_14_dfs.get('restaurant_14th_participant'), 
    participant_15_dfs.get('restaurant_15th_participant'), participant_16_dfs.get('restaurant_16th_participant'),
    participant_17_dfs.get('restaurant_17th_participant'), participant_18_dfs.get('restaurant_18th_participant'), 
    participant_19_dfs.get('restaurant_19th_participant'), participant_20_dfs.get('restaurant_20th_participant')
)
# Compute average rank 
restaurant_21_25_Female = add_average_rank(restaurant_21_25_Female, '21_25_Female_AR')

# Office Ranked Data for 10 Male participants between 21 and 25
office_21_25_Female = merge_participant_data(
    participant_11_dfs.get('office_11th_participant'), participant_12_dfs.get('office_12th_participant'), 
    participant_13_dfs.get('office_13th_participant'), participant_14_dfs.get('office_14th_participant'), 
    participant_15_dfs.get('office_15th_participant'), participant_16_dfs.get('office_16th_participant'),
    participant_17_dfs.get('office_17th_participant'), participant_18_dfs.get('office_18th_participant'), 
    participant_19_dfs.get('office_19th_participant'), participant_20_dfs.get('office_20th_participant')
)
# Compute average rank
office_21_25_Female = add_average_rank(office_21_25_Female, '21_25_Female_AR')

## Male Participant between age 26 - 30

In [29]:
basepath='/Users/mac/Documents/Music recommendation/26-30_Male'
participant_21_dfs = process_participant_data(21, basepath=basepath)
participant_22_dfs = process_participant_data(22, basepath=basepath)
participant_23_dfs = process_participant_data(23, basepath=basepath)
participant_24_dfs = process_participant_data(24, basepath=basepath)
participant_25_dfs = process_participant_data(25, basepath=basepath)
participant_26_dfs = process_participant_data(26, basepath=basepath)
participant_27_dfs = process_participant_data(27, basepath=basepath)
participant_28_dfs = process_participant_data(28, basepath=basepath)
participant_29_dfs = process_participant_data(29, basepath=basepath)
participant_30_dfs = process_participant_data(30, basepath=basepath)

# Bar Ranked Data for Male participant between 26 and 30
bar_26_30_Male = merge_participant_data(
    participant_21_dfs.get('bar_21st_participant'), participant_22_dfs.get('bar_22nd_participant'), 
    participant_23_dfs.get('bar_23rd_participant'), participant_24_dfs.get('bar_24th_participant'), 
    participant_25_dfs.get('bar_25th_participant'), participant_26_dfs.get('bar_26th_participant'),
    participant_27_dfs.get('bar_27th_participant'), participant_28_dfs.get('bar_28th_participant'), 
    participant_29_dfs.get('bar_29th_participant'), participant_30_dfs.get('bar_30th_participant')
)
# Compute average rank and drop the original ranking columns
bar_26_30_Male = add_average_rank(bar_26_30_Male, '26_30_Male_AR')

# Bedroom Ranked Data for 10 Male participants between 21 and 25
bedroom_26_30_Male = merge_participant_data(
    participant_21_dfs.get('bedroom_21st_participant'), participant_22_dfs.get('bedroom_22nd_participant'), 
    participant_23_dfs.get('bedroom_23rd_participant'), participant_24_dfs.get('bedroom_24th_participant'), 
    participant_25_dfs.get('bedroom_25th_participant'), participant_26_dfs.get('bedroom_26th_participant'),
    participant_27_dfs.get('bedroom_27th_participant'), participant_28_dfs.get('bedroom_28th_participant'), 
    participant_29_dfs.get('bedroom_29th_participant'), participant_30_dfs.get('bedroom_30th_participant')
)
# Compute average rank
bedroom_26_30_Male = add_average_rank(bedroom_26_30_Male, '26_30_Male_AR')

# Gym Ranked Data for 10 Male participants between 21 and 25
gym_26_30_Male = merge_participant_data(
    participant_21_dfs.get('gym_21st_participant'), participant_22_dfs.get('gym_22nd_participant'), 
    participant_23_dfs.get('gym_23rd_participant'), participant_24_dfs.get('gym_24th_participant'), 
    participant_25_dfs.get('gym_25th_participant'), participant_26_dfs.get('gym_26th_participant'),
    participant_27_dfs.get('gym_27th_participant'), participant_28_dfs.get('gym_28th_participant'), 
    participant_29_dfs.get('gym_29th_participant'), participant_30_dfs.get('gym_30th_participant')
)
# Compute average rank 
gym_26_30_Male = add_average_rank(gym_26_30_Male, '26_30_Male_AR')

# Mall Ranked Data for 10 Male participants between 21 and 25
mall_26_30_Male = merge_participant_data(
    participant_21_dfs.get('mall_21st_participant'), participant_22_dfs.get('mall_22nd_participant'), 
    participant_23_dfs.get('mall_23rd_participant'), participant_24_dfs.get('mall_24th_participant'), 
    participant_25_dfs.get('mall_25th_participant'), participant_26_dfs.get('mall_26th_participant'),
    participant_27_dfs.get('mall_27th_participant'), participant_28_dfs.get('mall_28th_participant'), 
    participant_29_dfs.get('mall_29th_participant'), participant_30_dfs.get('mall_30th_participant')
)
# Compute average rank 
mall_26_30_Male = add_average_rank(mall_26_30_Male, '26_30_Male_AR')

# Library Ranked Data for 10 Male participants between 21 and 25
library_26_30_Male = merge_participant_data(
    participant_21_dfs.get('library_21st_participant'), participant_22_dfs.get('library_22nd_participant'), 
    participant_23_dfs.get('library_23rd_participant'), participant_24_dfs.get('library_24th_participant'), 
    participant_25_dfs.get('library_25th_participant'), participant_26_dfs.get('library_26th_participant'),
    participant_27_dfs.get('library_27th_participant'), participant_28_dfs.get('library_28th_participant'), 
    participant_29_dfs.get('library_29th_participant'), participant_30_dfs.get('library_30th_participant')
)
# Compute average rank 
library_26_30_Male = add_average_rank(library_26_30_Male, '26_30_Male_AR')

# TrainBus Ranked Data for 10 Male participants between 21 and 25
trainbus_26_30_Male = merge_participant_data(
    participant_21_dfs.get('trainbus_21st_participant'), participant_22_dfs.get('trainbus_22nd_participant'), 
    participant_23_dfs.get('trainbus_23rd_participant'), participant_24_dfs.get('trainbus_24th_participant'), 
    participant_25_dfs.get('trainbus_25th_participant'), participant_26_dfs.get('trainbus_26th_participant'),
    participant_27_dfs.get('trainbus_27th_participant'), participant_28_dfs.get('trainbus_28th_participant'), 
    participant_29_dfs.get('trainbus_29th_participant'), participant_30_dfs.get('trainbus_30th_participant')
)
# Compute average rank
trainbus_26_30_Male = add_average_rank(trainbus_26_30_Male, '26_30_Male_AR')

# Restaurant Ranked Data for 10 Male participants between 21 and 25
restaurant_26_30_Male = merge_participant_data(
    participant_21_dfs.get('restaurant_21st_participant'), participant_22_dfs.get('restaurant_22nd_participant'), 
    participant_23_dfs.get('restaurant_23rd_participant'), participant_24_dfs.get('restaurant_24th_participant'), 
    participant_25_dfs.get('restaurant_25th_participant'), participant_26_dfs.get('restaurant_26th_participant'),
    participant_27_dfs.get('restaurant_27th_participant'), participant_28_dfs.get('restaurant_28th_participant'), 
    participant_29_dfs.get('restaurant_29th_participant'), participant_30_dfs.get('restaurant_30th_participant')
)
# Compute average rank 
restaurant_26_30_Male = add_average_rank(restaurant_26_30_Male, '26_30_Male_AR')

# Office Ranked Data for 10 Male participants between 21 and 25
office_26_30_Male = merge_participant_data(
    participant_21_dfs.get('office_21st_participant'), participant_22_dfs.get('office_22nd_participant'), 
    participant_23_dfs.get('office_23rd_participant'), participant_24_dfs.get('office_24th_participant'), 
    participant_25_dfs.get('office_25th_participant'), participant_26_dfs.get('office_26th_participant'),
    participant_27_dfs.get('office_27th_participant'), participant_28_dfs.get('office_28th_participant'), 
    participant_29_dfs.get('office_29th_participant'), participant_30_dfs.get('office_30th_participant')
)
# Compute average rank
office_26_30_Male = add_average_rank(office_26_30_Male, '26_30_Male_AR')

## Female Participant between age 26 - 30

In [30]:
basepath='/Users/mac/Documents/Music recommendation/26-30_Female'

participant_31_dfs = process_participant_data(31, basepath = basepath)
participant_32_dfs = process_participant_data(32, basepath = basepath)
participant_33_dfs = process_participant_data(33, basepath = basepath)
participant_34_dfs = process_participant_data(34, basepath = basepath)
participant_35_dfs = process_participant_data(35, basepath = basepath)
participant_36_dfs = process_participant_data(36, basepath = basepath)
participant_37_dfs = process_participant_data(37, basepath = basepath)
participant_38_dfs = process_participant_data(38, basepath = basepath)
participant_39_dfs = process_participant_data(39, basepath = basepath)
participant_40_dfs = process_participant_data(40, basepath = basepath)

# Bar Ranked Data for Male participant between 26 and 30
bar_26_30_Female = merge_participant_data(
    participant_31_dfs.get('bar_31st_participant'), participant_32_dfs.get('bar_32nd_participant'), 
    participant_33_dfs.get('bar_33rd_participant'), participant_34_dfs.get('bar_34th_participant'), 
    participant_35_dfs.get('bar_35th_participant'), participant_36_dfs.get('bar_36th_participant'),
    participant_37_dfs.get('bar_37th_participant'), participant_38_dfs.get('bar_38th_participant'), 
    participant_39_dfs.get('bar_39th_participant'), participant_40_dfs.get('bar_40th_participant')
)
# Compute average rank and drop the original ranking columns
bar_26_30_Female = add_average_rank(bar_26_30_Female, '26_30_Female_AR')

# Bedroom Ranked Data for 10 Male participants between 21 and 25
bedroom_26_30_Female = merge_participant_data(
    participant_31_dfs.get('bedroom_31st_participant'), participant_32_dfs.get('bedroom_32nd_participant'), 
    participant_33_dfs.get('bedroom_33rd_participant'), participant_34_dfs.get('bedroom_34th_participant'), 
    participant_35_dfs.get('bedroom_35th_participant'), participant_36_dfs.get('bedroom_36th_participant'),
    participant_37_dfs.get('bedroom_37th_participant'), participant_38_dfs.get('bedroom_38th_participant'), 
    participant_39_dfs.get('bedroom_39th_participant'), participant_40_dfs.get('bedroom_40th_participant')
)
# Compute average rank
bedroom_26_30_Female = add_average_rank(bedroom_26_30_Female, '26_30_Female_AR')

# Gym Ranked Data for 10 Male participants between 21 and 25
gym_26_30_Female = merge_participant_data(
    participant_31_dfs.get('gym_31st_participant'), participant_32_dfs.get('gym_32nd_participant'), 
    participant_33_dfs.get('gym_33rd_participant'), participant_34_dfs.get('gym_34th_participant'), 
    participant_35_dfs.get('gym_35th_participant'), participant_36_dfs.get('gym_36th_participant'),
    participant_37_dfs.get('gym_37th_participant'), participant_38_dfs.get('gym_38th_participant'), 
    participant_39_dfs.get('gym_39th_participant'), participant_40_dfs.get('gym_40th_participant')
)
# Compute average rank 
gym_26_30_Female = add_average_rank(gym_26_30_Female, '26_30_Female_AR')

# Mall Ranked Data for 10 Male participants between 21 and 25
mall_26_30_Female = merge_participant_data(
    participant_31_dfs.get('mall_31st_participant'), participant_32_dfs.get('mall_32nd_participant'), 
    participant_33_dfs.get('mall_33rd_participant'), participant_34_dfs.get('mall_34th_participant'), 
    participant_35_dfs.get('mall_35th_participant'), participant_36_dfs.get('mall_36th_participant'),
    participant_37_dfs.get('mall_37th_participant'), participant_38_dfs.get('mall_38th_participant'), 
    participant_39_dfs.get('mall_39th_participant'), participant_40_dfs.get('mall_40th_participant')
)
# Compute average rank 
mall_26_30_Female = add_average_rank(mall_26_30_Female, '26_30_Female_AR')

# Library Ranked Data for 10 Male participants between 21 and 25
library_26_30_Female = merge_participant_data(
    participant_31_dfs.get('library_31st_participant'), participant_32_dfs.get('library_32nd_participant'), 
    participant_33_dfs.get('library_33rd_participant'), participant_34_dfs.get('library_34th_participant'), 
    participant_35_dfs.get('library_35th_participant'), participant_36_dfs.get('library_36th_participant'),
    participant_37_dfs.get('library_37th_participant'), participant_38_dfs.get('library_38th_participant'), 
    participant_39_dfs.get('library_39th_participant'), participant_40_dfs.get('library_40th_participant')
)
# Compute average rank 
library_26_30_Female = add_average_rank(library_26_30_Female, '26_30_Female_AR')

# TrainBus Ranked Data for 10 Male participants between 21 and 25
trainbus_26_30_Female = merge_participant_data(
    participant_31_dfs.get('trainbus_31st_participant'), participant_32_dfs.get('trainbus_32nd_participant'), 
    participant_33_dfs.get('trainbus_33rd_participant'), participant_34_dfs.get('trainbus_34th_participant'), 
    participant_35_dfs.get('trainbus_35th_participant'), participant_36_dfs.get('trainbus_36th_participant'),
    participant_37_dfs.get('trainbus_37th_participant'), participant_38_dfs.get('trainbus_38th_participant'), 
    participant_39_dfs.get('trainbus_39th_participant'), participant_40_dfs.get('trainbus_40th_participant')
)
# Compute average rank
trainbus_26_30_Female = add_average_rank(trainbus_26_30_Female, '26_30_Female_AR')

# Restaurant Ranked Data for 10 Male participants between 21 and 25
restaurant_26_30_Female = merge_participant_data(
    participant_31_dfs.get('restaurant_31st_participant'), participant_32_dfs.get('restaurant_32nd_participant'), 
    participant_33_dfs.get('restaurant_33rd_participant'), participant_34_dfs.get('restaurant_34th_participant'), 
    participant_35_dfs.get('restaurant_35th_participant'), participant_36_dfs.get('restaurant_36th_participant'),
    participant_37_dfs.get('restaurant_37th_participant'), participant_38_dfs.get('restaurant_38th_participant'), 
    participant_39_dfs.get('restaurant_39th_participant'), participant_40_dfs.get('restaurant_40th_participant')
)
# Compute average rank 
restaurant_26_30_Female = add_average_rank(restaurant_26_30_Female, '26_30_Female_AR')

# Office Ranked Data for 10 Male participants between 21 and 25
office_26_30_Female = merge_participant_data(
    participant_31_dfs.get('office_31st_participant'), participant_32_dfs.get('office_32nd_participant'), 
    participant_33_dfs.get('office_33rd_participant'), participant_34_dfs.get('office_34th_participant'), 
    participant_35_dfs.get('office_35th_participant'), participant_36_dfs.get('office_36th_participant'),
    participant_37_dfs.get('office_37th_participant'), participant_38_dfs.get('office_38th_participant'), 
    participant_39_dfs.get('office_39th_participant'), participant_40_dfs.get('office_40th_participant')
)
# Compute average rank
office_26_30_Female = add_average_rank(office_26_30_Female, '26_30_Female_AR')

## Female Participant between age 31 - 35

In [31]:
basepath='/Users/mac/Documents/Music recommendation/31-35_Female'
participant_41_dfs = process_participant_data(41, basepath = basepath)
participant_42_dfs = process_participant_data(42, basepath = basepath)
participant_43_dfs = process_participant_data(43, basepath = basepath)
participant_44_dfs = process_participant_data(44, basepath = basepath)
participant_45_dfs = process_participant_data(45, basepath = basepath)
participant_46_dfs = process_participant_data(46, basepath = basepath)
participant_47_dfs = process_participant_data(47, basepath = basepath)
participant_48_dfs = process_participant_data(48, basepath = basepath)
participant_49_dfs = process_participant_data(49, basepath = basepath)
participant_50_dfs = process_participant_data(50, basepath = basepath)

# Bar Ranked Data for Female participant between 21 and 25
bar_31_35_Female = merge_participant_data(
    participant_41_dfs.get('bar_41st_participant'), participant_42_dfs.get('bar_42nd_participant'), 
    participant_43_dfs.get('bar_43rd_participant'), participant_44_dfs.get('bar_44th_participant'), 
    participant_45_dfs.get('bar_45th_participant'), participant_46_dfs.get('bar_46th_participant'),
    participant_47_dfs.get('bar_47th_participant'), participant_48_dfs.get('bar_48th_participant'), 
    participant_49_dfs.get('bar_49th_participant'), participant_50_dfs.get('bar_50th_participant')
)
# Compute average rank and drop the original ranking columns
bar_31_35_Female = add_average_rank(bar_31_35_Female, '31_35_Female_AR')

# Bedroom Ranked Data for 10 Female participants between 21 and 25
bedroom_31_35_Female = merge_participant_data(
    participant_41_dfs.get('bedroom_41st_participant'), participant_42_dfs.get('bedroom_42nd_participant'), 
    participant_43_dfs.get('bedroom_43rd_participant'), participant_44_dfs.get('bedroom_44th_participant'), 
    participant_45_dfs.get('bedroom_45th_participant'), participant_46_dfs.get('bedroom_46th_participant'),
    participant_47_dfs.get('bedroom_47th_participant'), participant_48_dfs.get('bedroom_48th_participant'), 
    participant_49_dfs.get('bedroom_49th_participant'), participant_50_dfs.get('bedroom_50th_participant')
)
# Compute average rank
bedroom_31_35_Female = add_average_rank(bedroom_31_35_Female, '31_35_Female_AR')

# Gym Ranked Data for 10 Female participants between 21 and 25
gym_31_35_Female = merge_participant_data(
    participant_41_dfs.get('gym_41st_participant'), participant_42_dfs.get('gym_42nd_participant'), 
    participant_43_dfs.get('gym_43rd_participant'), participant_44_dfs.get('gym_44th_participant'), 
    participant_45_dfs.get('gym_45th_participant'), participant_46_dfs.get('gym_46th_participant'),
    participant_47_dfs.get('gym_47th_participant'), participant_48_dfs.get('gym_48th_participant'), 
    participant_49_dfs.get('gym_49th_participant'), participant_50_dfs.get('gym_50th_participant')
)
# Compute average rank 
gym_31_35_Female = add_average_rank(gym_31_35_Female, '31_35_Female_AR')

# Mall Ranked Data for 10 Female participants between 21 and 25
mall_31_35_Female = merge_participant_data(
    participant_41_dfs.get('mall_41st_participant'), participant_42_dfs.get('mall_42nd_participant'), 
    participant_43_dfs.get('mall_43rd_participant'), participant_44_dfs.get('mall_44th_participant'), 
    participant_45_dfs.get('mall_45th_participant'), participant_46_dfs.get('mall_46th_participant'),
    participant_47_dfs.get('mall_47th_participant'), participant_48_dfs.get('mall_48th_participant'), 
    participant_49_dfs.get('mall_49th_participant'), participant_50_dfs.get('mall_50th_participant')
)
# Compute average rank 
mall_31_35_Female = add_average_rank(mall_31_35_Female, '31_35_Female_AR')

# Library Ranked Data for 10 Female participants between 21 and 25
library_31_35_Female = merge_participant_data(
    participant_41_dfs.get('library_41st_participant'), participant_42_dfs.get('library_42nd_participant'), 
    participant_43_dfs.get('library_43rd_participant'), participant_44_dfs.get('library_44th_participant'), 
    participant_45_dfs.get('library_45th_participant'), participant_46_dfs.get('library_46th_participant'),
    participant_47_dfs.get('library_47th_participant'), participant_48_dfs.get('library_48th_participant'), 
    participant_49_dfs.get('library_49th_participant'), participant_50_dfs.get('library_50th_participant')
)
# Compute average rank 
library_31_35_Female = add_average_rank(library_31_35_Female, '31_35_Female_AR')

# TrainBus Ranked Data for 10 Female participants between 21 and 25
trainbus_31_35_Female = merge_participant_data(
    participant_41_dfs.get('trainbus_41st_participant'), participant_42_dfs.get('trainbus_42nd_participant'), 
    participant_43_dfs.get('trainbus_43rd_participant'), participant_44_dfs.get('trainbus_44th_participant'), 
    participant_45_dfs.get('trainbus_45th_participant'), participant_46_dfs.get('trainbus_46th_participant'),
    participant_47_dfs.get('trainbus_47th_participant'), participant_48_dfs.get('trainbus_48th_participant'), 
    participant_49_dfs.get('trainbus_49th_participant'), participant_50_dfs.get('trainbus_50th_participant')
)
# Compute average rank
trainbus_31_35_Female = add_average_rank(trainbus_31_35_Female, '31_35_Female_AR')

# Restaurant Ranked Data for 10 Female participants between 21 and 25
restaurant_31_35_Female = merge_participant_data(
    participant_41_dfs.get('restaurant_41st_participant'), participant_42_dfs.get('restaurant_42nd_participant'), 
    participant_43_dfs.get('restaurant_43rd_participant'), participant_44_dfs.get('restaurant_44th_participant'), 
    participant_45_dfs.get('restaurant_45th_participant'), participant_46_dfs.get('restaurant_46th_participant'),
    participant_47_dfs.get('restaurant_47th_participant'), participant_48_dfs.get('restaurant_48th_participant'), 
    participant_49_dfs.get('restaurant_49th_participant'), participant_50_dfs.get('restaurant_50th_participant')
)
# Compute average rank 
restaurant_31_35_Female = add_average_rank(restaurant_31_35_Female, '31_35_Female_AR')

# Office Ranked Data for 10 Female participants between 21 and 25
office_31_35_Female = merge_participant_data(
    participant_41_dfs.get('office_41st_participant'), participant_42_dfs.get('office_42nd_participant'), 
    participant_43_dfs.get('office_43rd_participant'), participant_44_dfs.get('office_44th_participant'), 
    participant_45_dfs.get('office_45th_participant'), participant_46_dfs.get('office_46th_participant'),
    participant_47_dfs.get('office_47th_participant'), participant_48_dfs.get('office_48th_participant'), 
    participant_49_dfs.get('office_49th_participant'), participant_50_dfs.get('office_50th_participant')
)
# Compute average rank
office_31_35_Female = add_average_rank(office_31_35_Female, '31_35_Female_AR')

## Male Participant between age 31 - 35

In [32]:
basepath='/Users/mac/Documents/Music recommendation/31-35_Male'
participant_51_dfs = process_participant_data(51, basepath = basepath)
participant_52_dfs = process_participant_data(52, basepath = basepath)
participant_53_dfs = process_participant_data(53, basepath = basepath)
participant_54_dfs = process_participant_data(54, basepath = basepath)
participant_55_dfs = process_participant_data(55, basepath = basepath)
participant_56_dfs = process_participant_data(56, basepath = basepath)
participant_57_dfs = process_participant_data(57, basepath = basepath)
participant_58_dfs = process_participant_data(58, basepath = basepath)
participant_59_dfs = process_participant_data(59, basepath = basepath)
participant_60_dfs = process_participant_data(60, basepath = basepath)

# Bar Ranked Data for Male participant between 31 and 35
bar_31_35_Male = merge_participant_data(
    participant_51_dfs.get('bar_51st_participant'), participant_52_dfs.get('bar_52nd_participant'), 
    participant_53_dfs.get('bar_53rd_participant'), participant_54_dfs.get('bar_54th_participant'), 
    participant_55_dfs.get('bar_55th_participant'), participant_56_dfs.get('bar_56th_participant'),
    participant_57_dfs.get('bar_57th_participant'), participant_58_dfs.get('bar_58th_participant'), 
    participant_59_dfs.get('bar_59th_participant'), participant_60_dfs.get('bar_60th_participant')
)
# Compute average rank and drop the original ranking columns
bar_31_35_Male = add_average_rank(bar_31_35_Male, '31_35_Male_AR')

# Bedroom Ranked Data for 10 Male participants between 31 and 35
bedroom_31_35_Male = merge_participant_data(
    participant_51_dfs.get('bedroom_51st_participant'), participant_52_dfs.get('bedroom_52nd_participant'), 
    participant_53_dfs.get('bedroom_53rd_participant'), participant_54_dfs.get('bedroom_54th_participant'), 
    participant_55_dfs.get('bedroom_55th_participant'), participant_56_dfs.get('bedroom_56th_participant'),
    participant_57_dfs.get('bedroom_57th_participant'), participant_58_dfs.get('bedroom_58th_participant'), 
    participant_59_dfs.get('bedroom_59th_participant'), participant_60_dfs.get('bedroom_60th_participant')
)
# Compute average rank
bedroom_31_35_Male = add_average_rank(bedroom_31_35_Male, '31_35_Male_AR')

# Gym Ranked Data for 10 Male participants between 31 and 35
gym_31_35_Male = merge_participant_data(
    participant_51_dfs.get('gym_51st_participant'), participant_52_dfs.get('gym_52nd_participant'), 
    participant_53_dfs.get('gym_53rd_participant'), participant_54_dfs.get('gym_54th_participant'), 
    participant_55_dfs.get('gym_55th_participant'), participant_56_dfs.get('gym_56th_participant'),
    participant_57_dfs.get('gym_57th_participant'), participant_58_dfs.get('gym_58th_participant'), 
    participant_59_dfs.get('gym_59th_participant'), participant_60_dfs.get('gym_60th_participant')
)
# Compute average rank 
gym_31_35_Male = add_average_rank(gym_31_35_Male, '31_35_Male_AR')

# Mall Ranked Data for 10 Male participants between 31 and 35
mall_31_35_Male = merge_participant_data(
    participant_51_dfs.get('mall_51st_participant'), participant_52_dfs.get('mall_52nd_participant'), 
    participant_53_dfs.get('mall_53rd_participant'), participant_54_dfs.get('mall_54th_participant'), 
    participant_55_dfs.get('mall_55th_participant'), participant_56_dfs.get('mall_56th_participant'),
    participant_57_dfs.get('mall_57th_participant'), participant_58_dfs.get('mall_58th_participant'), 
    participant_59_dfs.get('mall_59th_participant'), participant_60_dfs.get('mall_60th_participant')
)
# Compute average rank 
mall_31_35_Male = add_average_rank(mall_31_35_Male, '31_35_Male_AR')

# TrainBus Ranked Data for 10 Male participants between 31 and 35
trainbus_31_35_Male = merge_participant_data(
    participant_51_dfs.get('trainbus_51st_participant'), participant_52_dfs.get('trainbus_52nd_participant'), 
    participant_53_dfs.get('trainbus_53rd_participant'), participant_54_dfs.get('trainbus_54th_participant'), 
    participant_55_dfs.get('trainbus_55th_participant'), participant_56_dfs.get('trainbus_56th_participant'),
    participant_57_dfs.get('trainbus_57th_participant'), participant_58_dfs.get('trainbus_58th_participant'), 
    participant_59_dfs.get('trainbus_59th_participant'), participant_60_dfs.get('trainbus_60th_participant')
)
# Compute average rank
trainbus_31_35_Male = add_average_rank(trainbus_31_35_Male, '31_35_Male_AR')


# Library Ranked Data for 10 Male participants between 31 and 35
library_31_35_Male = merge_participant_data(
    participant_51_dfs.get('library_51st_participant'), participant_52_dfs.get('library_52nd_participant'), 
    participant_53_dfs.get('library_53rd_participant'), participant_54_dfs.get('library_54th_participant'), 
    participant_55_dfs.get('library_55th_participant'), participant_56_dfs.get('library_56th_participant'),
    participant_57_dfs.get('library_57th_participant'), participant_58_dfs.get('library_58th_participant'), 
    participant_59_dfs.get('library_59th_participant'), participant_60_dfs.get('library_60th_participant')
)
# Compute average rank 
library_31_35_Male = add_average_rank(library_31_35_Male, '31_35_Male_AR')


# Restaurant Ranked Data for 10 Male participants between 31 and 35
restaurant_31_35_Male = merge_participant_data(
    participant_51_dfs.get('restaurant_51st_participant'), participant_52_dfs.get('restaurant_52nd_participant'), 
    participant_53_dfs.get('restaurant_53rd_participant'), participant_54_dfs.get('restaurant_54th_participant'), 
    participant_55_dfs.get('restaurant_55th_participant'), participant_56_dfs.get('restaurant_56th_participant'),
    participant_57_dfs.get('restaurant_57th_participant'), participant_58_dfs.get('restaurant_58th_participant'), 
    participant_59_dfs.get('restaurant_59th_participant'), participant_60_dfs.get('restaurant_60th_participant')
)
# Compute average rank 
restaurant_31_35_Male = add_average_rank(restaurant_31_35_Male, '31_35_Male_AR')

# Office Ranked Data for 10 Male participants between 21 and 25
office_31_35_Male = merge_participant_data(
    participant_51_dfs.get('office_51st_participant'), participant_52_dfs.get('office_52nd_participant'), 
    participant_53_dfs.get('office_53rd_participant'), participant_54_dfs.get('office_54th_participant'), 
    participant_55_dfs.get('office_55th_participant'), participant_56_dfs.get('office_56th_participant'),
    participant_57_dfs.get('office_57th_participant'), participant_58_dfs.get('office_58th_participant'), 
    participant_59_dfs.get('office_59th_participant'), participant_60_dfs.get('office_60th_participant')
)
# Compute average rank
office_31_35_Male = add_average_rank(office_31_35_Male, '31_35_Male_AR')

# Merge Data for the Different Demographic Group

In [7]:
# Define the DataFrames to merge
office_list = [office_26_30_Female, office_26_30_Male, office_21_25_Male, office_21_25_Female, office_31_35_Male, office_31_35_Female]
# Call the function to merge the DataFrames on 'S/N'
all_office_ranks = merge_dataframes_on_sn(office_list)

trainbus_list = [trainbus_26_30_Female, trainbus_26_30_Male, trainbus_21_25_Male, trainbus_21_25_Female, trainbus_31_35_Male, trainbus_31_35_Female]
# Call the function to merge the DataFrames on 'S/N'
all_trainbus_ranks = merge_dataframes_on_sn(trainbus_list)

library_list = [library_26_30_Female, library_26_30_Male, library_21_25_Male, library_21_25_Female, library_31_35_Male, library_31_35_Female]
# Call the function to merge the DataFrames on 'S/N'
all_library_ranks = merge_dataframes_on_sn(library_list)

# Define the DataFrames to merge
gym_list = [gym_26_30_Female, gym_26_30_Male, gym_21_25_Male, gym_21_25_Female, gym_31_35_Male, gym_31_35_Female]
# Call the function to merge the DataFrames on 'S/N'
all_gym_ranks = merge_dataframes_on_sn(gym_list)

bedroom_list = [bedroom_26_30_Female, bedroom_26_30_Male, bedroom_21_25_Male, bedroom_21_25_Female, bedroom_31_35_Male, bedroom_31_35_Female]
# Call the function to merge the DataFrames on 'S/N'
all_bedroom_ranks = merge_dataframes_on_sn(bedroom_list)

restaurant_list = [restaurant_26_30_Female, restaurant_26_30_Male, restaurant_21_25_Male, restaurant_21_25_Female, restaurant_31_35_Male, restaurant_31_35_Female]
# Call the function to merge the DataFrames on 'S/N'
all_restaurant_ranks = merge_dataframes_on_sn(restaurant_list)

bedroom_list = [bedroom_26_30_Female, bedroom_26_30_Male, bedroom_21_25_Male, bedroom_21_25_Female, bedroom_31_35_Male, bedroom_31_35_Female]
# Call the function to merge the DataFrames on 'S/N'
all_bedroom_ranks = merge_dataframes_on_sn(bedroom_list)

bar_list = [bar_26_30_Female, bar_26_30_Male, bar_21_25_Male, bar_21_25_Female, bar_31_35_Male, bar_31_35_Female]
# Call the function to merge the DataFrames on 'S/N'
all_bar_ranks = merge_dataframes_on_sn(bar_list)

mall_list = [mall_26_30_Female, mall_26_30_Male, mall_21_25_Male, mall_21_25_Female, mall_31_35_Male, mall_31_35_Female]
# Call the function to merge the DataFrames on 'S/N'
all_mall_ranks = merge_dataframes_on_sn(mall_list)

In [8]:
gym = pd.read_csv('gym_df.csv')
bedroom = pd.read_csv('bedroom_df.csv')
library = pd.read_csv('library_df.csv')
office = pd.read_csv('office_df.csv')
trainBus = pd.read_csv('trainbus_df.csv')
restaurant = pd.read_csv('restaurant_df.csv')
bar = pd.read_csv('bar_df.csv')
mall = pd.read_csv('mall_df.csv')

# Data Cleaning

In [9]:
# def normalize_text(text):
#     if isinstance(text, str):
#         text = text.lower()
#         text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')
#         return text.strip()
#     return text

# def find_special_character_indices(df):
#     """
#     Find indices of rows in the DataFrame where artist_name or track_name contains special characters.
    
#     Parameters:
#     df (pd.DataFrame): The DataFrame to check for special characters.
    
#     Returns:
#     list: List of indices where special characters are found.
#     """
#     special_char_pattern = re.compile(r'[^\x00-\x7F]+')

#     special_char_indices = df[
#         df['track_name'].apply(lambda x: bool(special_char_pattern.search(str(x)))) |
#         df['artist_name'].apply(lambda x: bool(special_char_pattern.search(str(x))))
#     ].index.tolist()
    
#     return special_char_indices

# def check_name(df, index):
#     """
#     Check the artist name and track name at a given index.

#     Parameters:
#     df (pd.DataFrame): The DataFrame containing the data.
#     index (int): The index to check.

#     Returns:
#     None
#     """
#     if index in df.index:
#         current_artist_name = df.loc[index, 'artist_name']
#         current_track_name = df.loc[index, 'track_name']
        
#         print(f"Current artist name at index {index}: {current_artist_name}")
#         print(f"Current track name at index {index}: {current_track_name}")
#     else:
#         print(f"Index {index} not found in the DataFrame.")

# def find_potential_matches(dataframe1, dataframe2, index, similarity_threshold=0.6):
#     """
#     Find potential matches for the artist and track name at a given index in dataframe1 within dataframe2.
    
#     Parameters:
#     dataframe1 (pd.DataFrame): The first DataFrame containing the name to match.
#     dataframe2 (pd.DataFrame): The second DataFrame to search for potential matches.
#     index (int): The index in dataframe1 to find potential matches for.
#     similarity_threshold (float): The threshold for similarity (0 to 1) for considering a match.
    
#     Returns:
#     list of tuples: A list of tuples with (index in dataframe2, similarity score).
#     """
#     if index in dataframe1.index:
#         artist_name_1 = dataframe1.loc[index, 'artist_name']
#         track_name_1 = dataframe1.loc[index, 'track_name']
        
#         potential_matches = []

#         for idx, row in dataframe2.iterrows():
#             artist_name_2 = row['artist_name']
#             track_name_2 = row['track_name']
            
#             artist_similarity = difflib.SequenceMatcher(None, artist_name_1, artist_name_2).ratio()
#             track_similarity = difflib.SequenceMatcher(None, track_name_1, track_name_2).ratio()
            
#             average_similarity = (artist_similarity + track_similarity) / 2
            
#             if average_similarity >= similarity_threshold:
#                 potential_matches.append((idx, average_similarity))

#         potential_matches.sort(key=lambda x: x[1], reverse=True)
        
#         return potential_matches
#     else:
#         return []

# def change_name(df, index, new_artist_name=None, new_track_name=None):
#     """
#     Change the artist name and track name at a given index.
    
#     Parameters:
#     df (pd.DataFrame): The DataFrame containing the data.
#     index (int): The index to change.
#     new_artist_name (str): The new artist name to update (optional).
#     new_track_name (str): The new track name to update (optional).
    
#     Returns:
#     None
#     """
#     if index in df.index:
#         if new_artist_name is not None:
#             df.loc[index, 'artist_name'] = new_artist_name
        
#         if new_track_name is not None:
#             df.loc[index, 'track_name'] = new_track_name

# def correct_special_character_names(dataframe1, dataframe2, similarity_threshold=0.6):
#     """
#     Correct names in dataframe1 that contain special characters by finding the best match in dataframe2.
    
#     Parameters:
#     dataframe1 (pd.DataFrame): The first DataFrame to correct.
#     dataframe2 (pd.DataFrame): The second DataFrame containing correct names.
#     similarity_threshold (float): The threshold for similarity (0 to 1) for considering a match.
    
#     Returns:
#     None: The changes are made directly to dataframe1.
#     """
#     special_char_indices = find_special_character_indices(dataframe1)
    
#     for index in special_char_indices:
#         print(f"Processing index {index} in dataframe1 with potential special characters:")
#         potential_matches = find_potential_matches(dataframe1, dataframe2, index, similarity_threshold)
        
#         if potential_matches:
#             best_match_index = potential_matches[0][0]
#             best_artist_name = dataframe2.loc[best_match_index, 'artist_name']
#             best_track_name = dataframe2.loc[best_match_index, 'track_name']
            
#             print(f"Best match found in dataframe2 at index {best_match_index}:")
#             print(f"  Artist Name: {best_artist_name}")
#             print(f"  Track Name: {best_track_name}")
            
#             # Update dataframe1 with the best match
#             change_name(dataframe1, index, new_artist_name=best_artist_name, new_track_name=best_track_name)
#             print(f"Updated dataframe1 at index {index}.\n")
#         else:
#             print(f"No suitable match found in dataframe2 for index {index}.\n")
# def convert_columns_to_string(df, columns=['artist_name', 'track_name']):
#     """
#     Convert specified columns in a DataFrame to string type.

#     Parameters:
#     df (pd.DataFrame): The DataFrame containing the columns to convert.
#     columns (list): A list of column names to convert to string type.

#     Returns:
#     pd.DataFrame: The DataFrame with specified columns converted to strings.
#     """
#     for column in columns:
#         df[column] = df[column].astype(str)
    
#     print(f"Converted columns {columns} to string in the DataFrame.")     

# Merge the dataframe of pre-processed participant data with the song metadata, musical features and lyrics

In [46]:


def merge_dataframes(df1, df2, on_columns, suffixes=('_df1', '_df2')):
    """
    Merge two DataFrames on specified columns with customizable suffixes for overlapping column names.

    Parameters:
    df1 (pd.DataFrame): The first DataFrame (e.g., dataframe1).
    df2 (pd.DataFrame): The second DataFrame (e.g., dataframe2).
    on_columns (list): The columns on which to merge the DataFrames.
    suffixes (tuple): A tuple of strings to append to overlapping column names in the result.

    Returns:
    pd.DataFrame: The merged DataFrame.
    """
    merged_df = pd.merge(
        df1,
        df2,
        on=on_columns,
        suffixes=suffixes
    )
    
    print(f"Merged DataFrame has {len(merged_df)} rows.")
    return merged_df

    
def remove_exact_duplicates(df):
    """
    Remove duplicate songs from the DataFrame where all specified features are identical.

    Parameters:
    df (pd.DataFrame): The DataFrame containing the song data.

    Returns:
    pd.DataFrame: The DataFrame with duplicate entries removed and the index reset.
    """
    # Define the columns to check for exact duplicates
    columns_to_check = [
        'artist_name', 'track_name'
    ]
    
    # Drop duplicates based on these columns, keeping the first occurrence
    df = df.drop_duplicates(subset=columns_to_check, keep='first')
    
    # Reset the index
    df = df.reset_index(drop=True)
    
    print(f"Duplicates removed. Number of rows after deduplication: {len(df)}")
    
    return df

pattern = re.compile(r"\b\d{2}/\d{2}/\d{2}\b|feat", re.IGNORECASE)

def match_pattern(lyrics):
    return bool(pattern.search(lyrics))


## TrainBus

In [47]:
# # correct_special_character_names(all_trainbus_ranks, trainBus, similarity_threshold=0.8)
# # convert_columns_to_string(all_trainbus_ranks, columns=['artist_name', 'track_name'])
# # convert_columns_to_string(trainBus, columns=['artist_name', 'track_name'])
# # check_name(all_trainbus_ranks, 368)

# # matches = find_potential_matches(all_trainbus_ranks, trainBus, 368)
# # print("Potential matches in dataframe2 (index, similarity score):")
# # for match in matches:
# #     print(f"Index: {match[0]}, Similarity: {match[1]:.2f}")
# #     check_name(trainBus, match[0])
    
# # change_name(all_trainbus_ranks, 368, new_artist_name="Mýa", new_track_name="My Love Is Like...Wo")
trainbus_rankMerged = merge_dataframes(all_trainbus_ranks, trainBus, on_columns=['artist_name', 'track_name'])

trainbus_rankMerged = remove_exact_duplicates(trainbus_rankMerged)

## Library

In [48]:
# # correct_special_character_names(all_library_ranks, library, similarity_threshold=0.8)

# # convert_columns_to_string(all_library_ranks, columns=['artist_name', 'track_name'])
# # convert_columns_to_string(library, columns=['artist_name', 'track_name'])

# # change_name(all_library_ranks, 459, new_artist_name="Li Zemin", new_track_name="Hope 希望")
# # change_name(all_library_ranks, 39, new_artist_name="Li Zemin", new_track_name="Slowly 慢慢地")

# #check_name(all_library_ranks, 39)
# #check_name_by_content(all_library_ranks, 'Hope Â∏åÊúõ', 'Li Zemin')

# #matches = find_potential_matches(all_library_ranks, library, 39)
# #print("Potential matches in dataframe2 (index, similarity score):")
# #for match in matches:
#     #print(f"Index: {match[0]}, Similarity: {match[1]:.2f}")
#     #check_name(library, match[0])
    
library_rankMerged = merge_dataframes(all_library_ranks, library, on_columns=['artist_name', 'track_name'])
#library_rankMerged[library_rankMerged.duplicated(subset=['artist_name', 'track_name'], keep=False)]
library_rankMerged=remove_exact_duplicates(library_rankMerged)

## Office

In [49]:
# correct_special_character_names(all_office_ranks, office, similarity_threshold=0.8)

# convert_columns_to_string(all_office_ranks, columns=['artist_name', 'track_name'])
# convert_columns_to_string(office, columns=['artist_name', 'track_name'])

# indices_to_drop = [34,608]  # Example indices to drop - frank ocean error and gord Bamford
# all_office_ranks = all_office_ranks.drop(index=indices_to_drop) 

# change_name(all_office_ranks, 45, new_artist_name="Global Communication", new_track_name="14:31")

#check_name(all_office_ranks, 654)
#check_name_by_content(all_office_ranks, 'Nice to Meet You', 'Gord Bamford')
#matches = find_potential_matches(all_office_ranks, office, 608)
#print("Potential matches in dataframe2 (index, similarity score):")
#for match in matches:
    #print(f"Index: {match[0]}, Similarity: {match[1]:.2f}")
    #check_name(office, match[0])
    
office_rankMerged = merge_dataframes(all_office_ranks, office, on_columns=['artist_name', 'track_name'])
#office_rankMerged[office_rankMerged.duplicated(subset=['artist_name', 'track_name'], keep=False)]

office_rankMerged=remove_exact_duplicates(office_rankMerged)

## Bar

In [50]:
# correct_special_character_names(all_bar_ranks, bar, similarity_threshold=0.8)

# convert_columns_to_string(all_bar_ranks, columns=['artist_name', 'track_name'])
# convert_columns_to_string(bar, columns=['artist_name', 'track_name'])

# change_name(all_bar_ranks, 46, new_artist_name="bülow", new_track_name="Get Stüpid")
# change_name(all_bar_ranks, 60, new_artist_name="Bazzi", new_track_name="3:15")

bar_rankMerged = merge_dataframes(all_bar_ranks, bar, on_columns=['artist_name', 'track_name'])

# find_missing_rows(all_bar_ranks, bar_rankMerged, df_name="all_bar_ranks")
#check_name(all_bar_ranks, 60)
#check_name_by_content(all_bar_ranks, '03:15:00', 'Bazzi')
#matches = find_potential_matches(all_bar_ranks, bar, 60)
#print("Potential matches in dataframe2 (index, similarity score):")
#for match in matches:
    #print(f"Index: {match[0]}, Similarity: {match[1]:.2f}")
    #check_name(bar, match[0])
    
#bar_rankMerged[bar_rankMerged.duplicated(subset=['artist_name', 'track_name'], keep=False)]
bar_rankMerged=remove_exact_duplicates(bar_rankMerged)

## Restaurant

In [51]:
# correct_special_character_names(all_restaurant_ranks, restaurant, similarity_threshold=0.8)
# convert_columns_to_string(all_restaurant_ranks, columns=['artist_name', 'track_name'])
# convert_columns_to_string(restaurant, columns=['artist_name', 'track_name'])

# change_name(all_restaurant_ranks, 323, new_artist_name="Anton√≠n Dvo≈ô√°k", new_track_name="Dvo≈ô√°k: 4 Romantic Pieces, Op. 75, B. 150: No. 1, Allegro moderato")
restaurant_rankMerged = merge_dataframes(all_restaurant_ranks, restaurant, on_columns=['artist_name', 'track_name'])
#find_missing_rows(all_restaurant_ranks, restaurant_rankMerged, df_name="all_restaurant_ranks")

restaurant_rankMerged=remove_exact_duplicates(restaurant_rankMerged)

## Gym

In [52]:
# convert_columns_to_string(all_gym_ranks, columns=['artist_name', 'track_name'])
# convert_columns_to_string(gym, columns=['artist_name', 'track_name'])

# correct_special_character_names(all_gym_ranks, gym, similarity_threshold=0.8)

# change_name(all_gym_ranks, 5, new_artist_name="¥$", new_track_name="CARNIVAL")
# change_name(all_gym_ranks, 26, new_artist_name="¥$", new_track_name="FUK SUMN")
# change_name(all_gym_ranks, 266, new_artist_name="Tiësto", new_track_name="10:35")

gym_rankMerged = merge_dataframes(all_gym_ranks, gym, on_columns=['artist_name', 'track_name'])

gym_rankMerged=remove_exact_duplicates(gym_rankMerged)

## Mall

In [53]:
# convert_columns_to_string(all_mall_ranks, columns=['artist_name', 'track_name'])
# convert_columns_to_string(mall, columns=['artist_name', 'track_name'])

# correct_special_character_names(all_mall_ranks, mall, similarity_threshold=0.8)

# change_name(all_mall_ranks, 66, new_artist_name="Tiësto", new_track_name="10:35")
# change_name(all_mall_ranks, 83, new_artist_name="Æj", new_track_name="Waking Up In Rome")

mall_rankMerged = merge_dataframes(all_mall_ranks, mall, on_columns=['artist_name', 'track_name'])

mall_rankMerged=remove_exact_duplicates(mall_rankMerged)

## Bedroom

In [54]:
# convert_columns_to_string(all_bedroom_ranks, columns=['artist_name', 'track_name'])
# convert_columns_to_string(bedroom, columns=['artist_name', 'track_name'])

# correct_special_character_names(all_bedroom_ranks, bedroom, similarity_threshold=0.8)

bedroom_rankMerged = merge_dataframes(all_bedroom_ranks, bedroom, on_columns=['artist_name', 'track_name'])

bedroom_rankMerged=remove_exact_duplicates(bedroom_rankMerged)

# Lyrics Preprocessing

## Lyrics that are not in English

As stated in my report, there are songs that are not in English and this would be a potential during mood classification because the BERT model were trained on English data, they will not understand text that are not in English.

To solve that, has_non_english_chars was created to detect text with more than 10 non English characters

In [27]:
# Compile the regex pattern to match any character that is not an English alphabet letter
pattern = re.compile(r"[^\x00-\x7F]", re.UNICODE)

def has_non_english_chars(lyrics):
    # Find all non-English characters
    matches = pattern.findall(lyrics)
    # Check if there are at least two non-English characters
    return len(matches) >= 10

# Apply the pattern to the entire 'lyrics' column and get the matching indices
#non_english_indices = bedroom_rankMerged[bedroom_rankMerged['lyrics'].apply(has_non_english_chars)].index.tolist()

#print("Indices of songs with at least two non-English characters:", non_english_indices)



Indices with dates in MM/DD/YY format or the word 'feat': [191, 226, 244, 388]


## More lyrics processing

The lyrics retrieved also contained artist name and after exploring the lyrics, I realised that the actual lyrics does start till the first appearance of the word "Lyrics" , so I removed all the words before the first occurence of Lyrics and the word "Lyrics" itself. I gave an exception to song categorised as "Lyrics not found"

In [28]:
def remove_text_before_lyrics(df, lyrics_column='lyrics'):
    # Function to remove text before the first occurrence of "Lyrics"
    def clean_lyrics_text(text):
        # Check if the text contains "Lyrics not found"
        if "Lyrics not found" in text:
            return text
        
        # Find the index of the first occurrence of "Lyrics"
        index = text.find("Lyrics")
        if index != -1:
            # Slice the text from the occurrence of "Lyrics"
            return text[index + len("Lyrics"):].strip()
        return text  # If "Lyrics" is not found, return the text as is

    # Apply the cleaning function to the specified column
    df['cleaned_lyrics'] = df[lyrics_column].apply(clean_lyrics_text)
    
    return df


remove_text_before_lyrics(bedroom_rankMerged, lyrics_column='lyrics')
remove_text_before_lyrics(bar_rankMerged, lyrics_column='lyrics')
remove_text_before_lyrics(mall_rankMerged, lyrics_column='lyrics')
remove_text_before_lyrics(library_rankMerged, lyrics_column='lyrics')
remove_text_before_lyrics(restaurant_rankMerged, lyrics_column='lyrics')
remove_text_before_lyrics(gym_rankMerged, lyrics_column='lyrics')
remove_text_before_lyrics(office_rankMerged, lyrics_column='lyrics')
remove_text_before_lyrics(trainbus_rankMerged, lyrics_column='lyrics')

,S/N,track_name,artist_name,26_30_Female_AR,26_30_Male_AR,21_25_Male_AR,21_25_Female_AR,release_date,popularity,duration_ms,...,mode,speechiness,acousticness,liveness,valence,tempo,track_name_count,artist_id,genre,cleaned_lyrics
0,1,SYRGAS (Till Anna) - J.O.X Remix,LBSB,2.1,1.3,2.9,4.4,03/11/2023,44,160216,...,0,0.3720,0.045100,0.3810,0.342,92.380,0,2iUVnGKQclQfYjdHGSZcX0,['swedish eurodance'],Lyrics not found
1,2,FUCKING RAGGARBIL,FLÄDERPOJKARNA,2.1,1.3,2.9,4.4,22/09/2023,34,144214,...,1,0.0731,0.221000,0.1420,0.304,139.885,0,6kVG7vRsL7U7UuATIAyPtd,['epa dunk'],Lyrics not found
2,3,TA EN ÖL (DOH DAH),Maximus,1.4,1.3,2.5,3.7,01/12/2023,32,141629,...,1,0.0779,0.009160,0.2510,0.816,144.955,0,7KXt61GtLmrIizwZkpZa5C,['swedish pop rap'],Lyrics not found
3,4,Hatar mig,Hugge,1.8,1.3,2.7,3.7,10/11/2023,34,116647,...,1,0.2750,0.000755,0.0775,0.686,170.038,2,1U17tqE6Kaphway5gmRD9g,"['epa dunk', 'russelater']",Lyrics not found
4,5,LAVISH 2.0,PERZZONA,2.1,1.3,3.2,3.7,29/12/2023,36,134226,...,1,0.2110,0.101000,0.4150,0.506,109.802,0,4WId87hZzznRDnqwQQLyeC,['epa dunk'],Lyrics not found
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
438,440,99 Problems,JAY-Z,2.6,4.2,3.7,4.2,14/11/2003,60,234626,...,0,0.3980,0.006610,0.1030,0.548,89.554,0,3nFkdlSjzX9mRTtwJOzDYB,"['east coast hip hop', 'gangster rap', 'hip ho...",for '99 Problems' by JAY-Z:\n\n227 Contributor...
439,441,Temperature,Sean Paul,3.6,3.5,3.2,2.4,27/09/2005,76,218573,...,0,0.0685,0.106000,0.0712,0.822,125.040,7,3Isy6kedDrgPYoTS1dazA9,"['dance pop', 'dancehall', 'pop', 'pop rap']",[Intro]\nDi gyaldem Schillaci\nChanderpaul\nSu...
440,442,La Tortura (feat. Alejandro Sanz),Shakira,4.0,3.1,2.8,2.2,03/06/2005,71,212893,...,1,0.0427,0.029700,0.1230,0.812,100.011,0,0EmeFodog0BfCgMzAIvKQp,"['colombian pop', 'dance pop', 'latin pop', 'p...",not in English
441,443,All Falls Down,Kanye West,3.8,4.7,3.4,2.4,13/02/2004,0,223506,...,0,0.5060,0.184000,0.1520,0.460,90.893,21,5K4W6rqBFWDnAN6FQUkS6x,"['chicago rap', 'hip hop', 'rap']","[Chorus: Syleena Johnson & Kanye West]\nOh, wh..."


# Split Lyrics by 100 word count

As stated in disertation report, music can convey varying mood throughout the song. So to account for this, I splitted the lyrics into sections with each seaction containing at most 100 words.

In [29]:

def split_lyrics_by_word_count(df, lyrics_column='lyrics', words_per_section=100):
    """
    Splits the lyrics into sections based on a specified word count.
    
    Parameters:
    - df: DataFrame containing the lyrics
    - lyrics_column: The column in the DataFrame where the lyrics are stored
    - words_per_section: The number of words each section should have
    
    Returns:
    - DataFrame with lyrics split into sections
    """

    def split_text(text, word_limit):
        words = text.split()
        sections = []
        for i in range(0, len(words), word_limit):
            section = " ".join(words[i:i + word_limit])
            sections.append(section)
        return sections
    
    # Apply the function to each row in the DataFrame
    df[lyrics_column + '_sections'] = df[lyrics_column].apply(lambda x: split_text(x, words_per_section))
    
    # Create separate columns for each section
    df_expanded = df[lyrics_column + '_sections'].apply(pd.Series)
    
    # Add the expanded columns to the original DataFrame
    df_expanded.columns = [f'Section_{i+1}' for i in range(df_expanded.shape[1])]
    df = pd.concat([df, df_expanded], axis=1)
    
    df = df.reindex(sorted(df.columns), axis=1)

    return df

In [30]:
# Split lyrics for all the locations

bedroom_rankMerged_split = split_lyrics_by_word_count(bedroom_rankMerged, lyrics_column='cleaned_lyrics', words_per_section=100)
mall_rankMerged_split = split_lyrics_by_word_count(mall_rankMerged, lyrics_column='cleaned_lyrics', words_per_section=100)
gym_rankMerged_split = split_lyrics_by_word_count(gym_rankMerged, lyrics_column='cleaned_lyrics', words_per_section=100)
restaurant_rankMerged_split = split_lyrics_by_word_count(restaurant_rankMerged, lyrics_column='cleaned_lyrics', words_per_section=100)
bar_rankMerged_split = split_lyrics_by_word_count(bar_rankMerged, lyrics_column='cleaned_lyrics', words_per_section=100)
office_rankMerged_split = split_lyrics_by_word_count(office_rankMerged, lyrics_column='cleaned_lyrics', words_per_section=100)
library_rankMerged_split = split_lyrics_by_word_count(library_rankMerged, lyrics_column='cleaned_lyrics', words_per_section=100)
trainbus_rankMerged_split = split_lyrics_by_word_count(trainbus_rankMerged, lyrics_column='cleaned_lyrics', words_per_section=100)

# Exploratory Data Analysis of Lyrics

In [34]:
def count_lyrics_categories(df, lyrics_column='lyrics'):
    """
    Counts the number of occurrences for 'lyrics not found', 'lyrics not in English',
    and all other lyrics in the specified column of the DataFrame.

    Parameters:
    - df: pandas DataFrame containing the lyrics data
    - lyrics_column: The column name where the lyrics status is stored

    Returns:
    - A dictionary with the counts of 'lyrics not found', 'lyrics not in English',
      and all other lyrics.
    """

    # Count of 'lyrics not found'
    lyrics_not_found_count = (df[lyrics_column] == 'Lyrics not found').sum()

    # Count of 'lyrics not in English'
    lyrics_not_in_english_count = (df[lyrics_column] == 'Lyrics not in English').sum()

    # Count of all other lyrics
    other_lyrics_count = (~df[lyrics_column].isin(['Lyrics not found', 'Lyrics not in English'])).sum()

    return {
        'Lyrics_not_found': lyrics_not_found_count,
        'Lyrics_not_in_english': lyrics_not_in_english_count,
        'Other_lyrics': other_lyrics_count
    }

In [37]:
counts = count_lyrics_categories(bedroom_rankMerged_split)
print('Bedroom:',counts)

Bedroom: {'Lyrics_not_found': 222, 'Lyrics_not_in_english': 13, 'Other_lyrics': 535}


In [177]:
counts = count_lyrics_categories(mall_rankMerged_split)
print('Mall:',counts)

Mall: {'Lyrics_not_found': 95, 'Lyrics_not_in_english': 6, 'Other_lyrics': 136}


In [178]:
counts = count_lyrics_categories(gym_rankMerged_split)
print('Gym:',counts)

Gym: {'Lyrics_not_found': 145, 'Lyrics_not_in_english': 18, 'Other_lyrics': 428}


In [179]:
counts = count_lyrics_categories(restaurant_rankMerged_split)
print('Restaurant:',counts)

Restaurant: {'Lyrics_not_found': 113, 'Lyrics_not_in_english': 6, 'Other_lyrics': 458}


In [180]:
counts = count_lyrics_categories(bar_rankMerged_split)
print('Bar:',counts)

Bar: {'Lyrics_not_found': 147, 'Lyrics_not_in_english': 7, 'Other_lyrics': 329}


In [181]:
counts = count_lyrics_categories(office_rankMerged_split)
print('Office:',counts)

Office: {'Lyrics_not_found': 115, 'Lyrics_not_in_english': 75, 'Other_lyrics': 538}


In [182]:
counts = count_lyrics_categories(library_rankMerged_split)
print('Library:',counts)

Library: {'Lyrics_not_found': 304, 'Lyrics_not_in_english': 20, 'Other_lyrics': 139}


In [183]:
counts = count_lyrics_categories(trainbus_rankMerged_split)
print('Trainbus:',counts)

Trainbus: {'Lyrics_not_found': 97, 'Lyrics_not_in_english': 28, 'Other_lyrics': 318}


# Modelling - Check Lyrics Mood Modellling Notebook

# Download Data After Modelling

After modelling, download the data and continue preprocessing of the data.

In [ ]:
gym_rankMerged_split = pd.read_csv('gym_dataset.csv')
bedroom_rankMerged_split = pd.read_csv('bedroom_dataset.csv')
library_rankMerged_split = pd.read_csv('library_dataset.csv')
office_rankMerged_split = pd.read_csv('office_dataset.csv')
trainbus_rankMerged_split = pd.read_csv('trainbus_dataset.csv')
restaurant_rankMerged_split = pd.read_csv('restaurant_dataset.csv')
bar_rankMerged_split = pd.read_csv('bar_dataset.csv')
mall_rankMerged_split = pd.read_csv('mall_dataset.csv')

## Although is not important, I rearranged the names of the columns I have in my data. I did this manually because the number of lyrics section is different for each location

In [3]:
order = ['S/N', 'track_id', 'track_name', 'artist_name', 'artist_id', 'release_date', 
         'genre', 'key', 'liveness', 'loudness', 'mode', 'popularity', 'acousticness',
         'speechiness', 'tempo', 'valence', 'danceability', 'duration_ms', 'energy', 
         'lyrics', 'cleaned_lyrics', 'Section_1', 'Section_2', 'Section_3', 'Section_4', 
         'Section_5', 'Section_6', 'Section_7', 'Section_8', 'Section_9', 'Section_10',
         'section_moods', 'section_mood_mitch', 'section_mood_jhart', 
         '21_25_Female_AR', '21_25_Male_AR', '26_30_Female_AR', '26_30_Male_AR', '31_35_Male_AR', '31_35_Female_AR']

order2 = ['S/N', 'track_id', 'track_name', 'artist_name', 'artist_id', 'release_date', 
          'genre', 'key', 'liveness', 'loudness', 'mode', 'popularity', 'acousticness',
          'speechiness', 'tempo', 'valence', 'danceability', 'duration_ms', 'energy', 
          'lyrics', 'cleaned_lyrics', 'Section_1', 'Section_2', 'Section_3', 'Section_4', 
          'Section_5', 'Section_6', 'Section_7', 'Section_8', 'Section_9', 'Section_10', 'Section_11',
          'section_moods', 'section_mood_mitch', 'section_mood_jhart', 
          '21_25_Female_AR', '21_25_Male_AR', '26_30_Female_AR', '26_30_Male_AR', '31_35_Male_AR', '31_35_Female_AR']

order3 = ['S/N', 'track_id', 'track_name', 'artist_name', 'artist_id', 'release_date', 
          'genre', 'key', 'liveness', 'loudness', 'mode', 'popularity', 'acousticness',
          'speechiness', 'tempo', 'valence', 'danceability', 'duration_ms', 'energy', 
          'lyrics', 'cleaned_lyrics', 'Section_1', 'Section_2', 'Section_3', 'Section_4', 
          'Section_5', 'Section_6', 'Section_7', 'Section_8', 'Section_9', 'Section_10', 
          'Section_11', 'Section_12', 'Section_13', 'Section_14',
          'section_moods', 'section_mood_mitch', 'section_mood_jhart', 
          '21_25_Female_AR', '21_25_Male_AR', '26_30_Female_AR', '26_30_Male_AR', '31_35_Male_AR', '31_35_Female_AR']

order4 = ['S/N', 'track_id', 'track_name', 'artist_name', 'artist_id', 'release_date', 
          'genre', 'key', 'liveness', 'loudness', 'mode', 'popularity', 'acousticness',
          'speechiness', 'tempo', 'valence', 'danceability', 'duration_ms', 'energy', 
          'lyrics', 'cleaned_lyrics', 'Section_1', 'Section_2', 'Section_3', 'Section_4', 
          'Section_5', 'Section_6', 'Section_7', 'Section_8', 'Section_9', 'Section_10', 
          'Section_11', 'Section_12', 'Section_13', 'Section_14', 'Section_15',
          'section_moods', 'section_mood_mitch', 'section_mood_jhart', 
          '21_25_Female_AR', '21_25_Male_AR', '26_30_Female_AR', '26_30_Male_AR', '31_35_Male_AR', '31_35_Female_AR']

order5 = ['S/N', 'track_id', 'track_name', 'artist_name', 'artist_id', 'release_date', 
          'genre', 'key', 'liveness', 'loudness', 'mode', 'popularity', 'acousticness',
          'speechiness', 'tempo', 'valence', 'danceability', 'duration_ms', 'energy', 
          'lyrics', 'cleaned_lyrics', 'Section_1', 'Section_2', 'Section_3', 'Section_4', 
          'Section_5', 'Section_6',
          'section_moods', 'section_mood_mitch', 'section_mood_jhart', 
          '21_25_Female_AR', '21_25_Male_AR', '26_30_Female_AR', '26_30_Male_AR', '31_35_Male_AR', '31_35_Female_AR']


gym_rankMerged_split = gym_rankMerged_split[order]
bedroom_rankMerged_split = bedroom_rankMerged_split[order]
library_rankMerged_split = library_rankMerged_split[order2]
office_rankMerged_split = office_rankMerged_split[order3]
trainbus_rankMerged_split = trainbus_rankMerged_split[order4]
restaurant_rankMerged_split = restaurant_rankMerged_split[order]
bar_rankMerged_split = bar_rankMerged_split[order]
mall_rankMerged_split = mall_rankMerged_split[order5]

## I mistakenly took out the lyrics in Lyrics not in English when I called the remove_text_before_lyrics function without accounting for lyric not in English category and it affected it. The code below remedied it.

In [5]:


bedroom_rankMerged_split.loc[bedroom_rankMerged_split['cleaned_lyrics'] == 'not in English',
                             'cleaned_lyrics'] = 'Lyrics not in English'

gym_rankMerged_split.loc[gym_rankMerged_split['cleaned_lyrics'] == 'not in English',
                             'cleaned_lyrics'] = 'Lyrics not in English'

trainbus_rankMerged_split.loc[trainbus_rankMerged_split['cleaned_lyrics'] == 'not in English',
                              'cleaned_lyrics'] = 'Lyrics not in English'

library_rankMerged_split.loc[library_rankMerged_split['cleaned_lyrics'] == 'not in English',
                              'cleaned_lyrics'] = 'Lyrics not in English'

restaurant_rankMerged_split.loc[restaurant_rankMerged_split['cleaned_lyrics'] == 'not in English',
                                'cleaned_lyrics'] = 'Lyrics not in English'

office_rankMerged_split.loc[office_rankMerged_split['cleaned_lyrics'] == 'not in English',
                              'cleaned_lyrics'] = 'Lyrics not in English'

bar_rankMerged_split.loc[bar_rankMerged_split['cleaned_lyrics'] == 'not in English',
                              'cleaned_lyrics'] = 'Lyrics not in English'

mall_rankMerged_split.loc[mall_rankMerged_split['cleaned_lyrics'] == 'not in English',
                              'cleaned_lyrics'] = 'Lyrics not in English'

## I had to update the songs that are categorised as  Lyrics not in English and Lyrics not found to be the name of the mood in the three mood columns this because during modelling, the BERT models stated them that as neutral mood. Although this correct, it doesn't represent the semantic of the song data because that is likely not the mood of the song


Please note, these columns and what they mean:

* section_moods: Roberta classified mood
* section_mood_mitch: Distilroberta Mitch classified mood
* section_mood_jhart: Distilroberta classified mood

In [6]:
def update_section_moods(df):
    """
    This function updates the 'section_moods', 'section_mood_mitch', 
    and 'section_mood_jhart' columns in the given DataFrame based on the
    values in 'lyrics' and 'cleaned_lyrics'. If 'lyrics' or 'cleaned_lyrics'
    contain 'Lyrics not found' or 'Lyrics not in English', the corresponding
    'section_moods', 'section_mood_mitch', and 'section_mood_jhart' columns 
    will be updated with the same value.
    
    Parameters:
    df (pd.DataFrame): The DataFrame to update.
    
    Returns:
    None: The DataFrame is updated in place.
    """
    
    # Define the conditions where the update should occur
    conditions = (df['lyrics'].isin(['Lyrics not found', 'Lyrics not in English']) | 
                  df['cleaned_lyrics'].isin(['Lyrics not found', 'Lyrics not in English']))

    # Apply the update to the specified columns
    df.loc[conditions, ['section_moods', 'section_mood_mitch', 'section_mood_jhart']] = df['lyrics']


In [7]:
update_section_moods(bedroom_rankMerged_split)
update_section_moods(gym_rankMerged_split)
update_section_moods(trainbus_rankMerged_split)
update_section_moods(library_rankMerged_split)
update_section_moods(restaurant_rankMerged_split)
update_section_moods(office_rankMerged_split)
update_section_moods(bar_rankMerged_split)
update_section_moods(mall_rankMerged_split)

## Following the approach used by Cheng and Shen (2016), the mood of the song is the most frequent mood from all the sections

In [8]:
from collections import Counter
import ast

def get_frequent_mood(df, source_column, new_column):
    """
    Processes the source_column in df to determine the most frequent mood.
    
    - If the value is 'Lyrics not found' or 'Lyrics not in English', sets new_column to that value.
    - If the value is a list:
        - Removes None values.
        - If the list is empty, sets new_column to None.
        - If there is a single most frequent mood, sets new_column to that mood.
        - If there is a tie, sets new_column to the list of moods and records the row index.
    
    Parameters:
        df (pd.DataFrame): The DataFrame to process.
        source_column (str): The name of the column to process.
        new_column (str): The name of the column where results will be stored.
    
    Returns:
        list: A list of row indices where there was a tie for the most frequent mood.
    """
    tie_indices = []

    def determine_value(item, index):
        # Handle special strings
        if item in ['Lyrics not found', 'Lyrics not in English']:
            return item
        
        # Convert string representations of lists to actual lists
        if isinstance(item, str):
            try:
                # Safely evaluate string to list
                item = ast.literal_eval(item)
            except (ValueError, SyntaxError):
                # If conversion fails, return None
                return None
        
        if isinstance(item, list):
            # Filter out None values
            filtered_list = [mood for mood in item if mood is not None]
            
            if not filtered_list:
                return None
            
            # Count occurrences of each mood
            mood_counts = Counter(filtered_list)
            most_common = mood_counts.most_common()
            
            # Get the highest frequency
            highest_freq = most_common[0][1]
            
            # Find all moods that have the highest frequency
            top_moods = [mood for mood, count in most_common if count == highest_freq]
            
            if len(top_moods) == 1:
                return top_moods[0]
            else:
                tie_indices.append(index)
                return top_moods
        else:
            return None

    # Apply the function to each row
    df[new_column] = df.apply(lambda row: determine_value(row[source_column], row.name), axis=1)
    
    return tie_indices

In [9]:
bedroom_tie_rows = get_frequent_mood(bedroom_rankMerged_split, source_column='section_moods',
                                     new_column='final_mood_roberta')
gym_tie_rows = get_frequent_mood(gym_rankMerged_split, source_column='section_moods', 
                                     new_column='final_mood_roberta')
trainbus_tie_rows = get_frequent_mood(trainbus_rankMerged_split, source_column='section_moods',
                                     new_column='final_mood_roberta')
library_tie_rows = get_frequent_mood(library_rankMerged_split, source_column='section_moods', 
                                     new_column='final_mood_roberta')
restaurant_tie_rows = get_frequent_mood(restaurant_rankMerged_split, source_column='section_moods',
                                     new_column='final_mood_roberta')
office_tie_rows = get_frequent_mood(office_rankMerged_split, source_column='section_moods', 
                                     new_column='final_mood_roberta')
bar_tie_rows = get_frequent_mood(bar_rankMerged_split, source_column='section_moods',
                                     new_column='final_mood_roberta')
mall_tie_rows = get_frequent_mood(mall_rankMerged_split, source_column='section_moods', 
                                     new_column='final_mood_roberta')

In [10]:
bedroom_tie_rows2 = get_frequent_mood(bedroom_rankMerged_split, source_column='section_mood_mitch',
                                     new_column='final_mood_mitch')
gym_tie_rows2 = get_frequent_mood(gym_rankMerged_split, source_column='section_mood_mitch', 
                                     new_column='final_mood_mitch')
trainbus_tie_rows2 = get_frequent_mood(trainbus_rankMerged_split, source_column='section_mood_mitch',
                                     new_column='final_mood_mitch')
library_tie_rows2 = get_frequent_mood(library_rankMerged_split, source_column='section_mood_mitch', 
                                     new_column='final_mood_mitch')
restaurant_tie_rows2 = get_frequent_mood(restaurant_rankMerged_split, source_column='section_mood_mitch',
                                     new_column='final_mood_mitch')
office_tie_rows2 = get_frequent_mood(office_rankMerged_split, source_column='section_mood_mitch', 
                                     new_column='final_mood_mitch')
bar_tie_rows2 = get_frequent_mood(bar_rankMerged_split, source_column='section_mood_mitch',
                                     new_column='final_mood_mitch')
mall_tie_rows2 = get_frequent_mood(mall_rankMerged_split, source_column='section_mood_mitch', 
                                     new_column='final_mood_mitch')

In [11]:
bedroom_tie_rows3 = get_frequent_mood(bedroom_rankMerged_split, source_column='section_mood_jhart',
                                     new_column='final_mood_jhart')
gym_tie_rows3 = get_frequent_mood(gym_rankMerged_split, source_column='section_mood_jhart', 
                                     new_column='final_mood_jhart')
trainbus_tie_rows3 = get_frequent_mood(trainbus_rankMerged_split, source_column='section_mood_jhart',
                                     new_column='final_mood_jhart')
library_tie_rows3 = get_frequent_mood(library_rankMerged_split, source_column='section_mood_jhart', 
                                     new_column='final_mood_jhart')
restaurant_tie_rows3 = get_frequent_mood(restaurant_rankMerged_split, source_column='section_mood_jhart',
                                     new_column='final_mood_jhart')
office_tie_rows3 = get_frequent_mood(office_rankMerged_split, source_column='section_mood_jhart', 
                                     new_column='final_mood_jhart')
bar_tie_rows3 = get_frequent_mood(bar_rankMerged_split, source_column='section_mood_jhart',
                                     new_column='final_mood_jhart')
mall_tie_rows3 = get_frequent_mood(mall_rankMerged_split, source_column='section_mood_jhart', 
                                     new_column='final_mood_jhart')

In [12]:
print('Tie moods in bedroom for the roberta model:',len(bedroom_tie_rows))
print('Tie moods in gym for the roberta model:',len(gym_tie_rows))
print('Tie moods in trainbus for the roberta model:',len(trainbus_tie_rows))
print('Tie moods in library for the roberta model:',len(library_tie_rows))
print('Tie moods in restaurant for the roberta model:',len(restaurant_tie_rows))
print('Tie moods in office for the roberta model:',len(office_tie_rows))
print('Tie moods in bar for the roberta model:',len(bar_tie_rows))
print('Tie moods in mall for the roberta model:',len(mall_tie_rows))

Tie moods in bedroom for the roberta model: 101
Tie moods in gym for the roberta model: 43
Tie moods in trainbus for the roberta model: 47
Tie moods in library for the roberta model: 19
Tie moods in restaurant for the roberta model: 89
Tie moods in office for the roberta model: 85
Tie moods in bar for the roberta model: 31
Tie moods in mall for the roberta model: 29


In [13]:
print('Tie moods in bedroom for the distilroberta (mitch) model:',len(bedroom_tie_rows2))
print('Tie moods in gym for the distilroberta (mitch) model:',len(gym_tie_rows2))
print('Tie moods in trainbus for the distilroberta (mitch) model:',len(trainbus_tie_rows2))
print('Tie moods in library for the distilroberta (mitch) model:',len(library_tie_rows2))
print('Tie moods in restaurant for the distilroberta (mitch) model:',len(restaurant_tie_rows2))
print('Tie moods in office for the distilroberta (mitch) model:',len(office_tie_rows2))
print('Tie moods in bar for the distilroberta (mitch) model:',len(bar_tie_rows2))
print('Tie moods in mall for the distilroberta (mitch) model:',len(mall_tie_rows2))

Tie moods in bedroom for the distilroberta (mitch) model: 101
Tie moods in gym for the distilroberta (mitch) model: 56
Tie moods in trainbus for the distilroberta (mitch) model: 55
Tie moods in library for the distilroberta (mitch) model: 27
Tie moods in restaurant for the distilroberta (mitch) model: 98
Tie moods in office for the distilroberta (mitch) model: 85
Tie moods in bar for the distilroberta (mitch) model: 52
Tie moods in mall for the distilroberta (mitch) model: 23


In [14]:
print('Tie moods in bedroom for the distilroberta (jhart) model:',len(bedroom_tie_rows3))
print('Tie moods in gym for the distilroberta (jhart) model:',len(gym_tie_rows3))
print('Tie moods in trainbus for the distilroberta (jhart) model:',len(trainbus_tie_rows3))
print('Tie moods in library for the distilroberta (jhart) model:',len(library_tie_rows3))
print('Tie moods in restaurant for the distilroberta (jhart) model:',len(restaurant_tie_rows3))
print('Tie moods in office for the distilroberta (jhart) model:',len(office_tie_rows3))
print('Tie moods in bar for the distilroberta (jhart) model:',len(bar_tie_rows3))
print('Tie moods in mall for the distilroberta (jhart) model:',len(mall_tie_rows3))

Tie moods in bedroom for the distilroberta (jhart) model: 142
Tie moods in gym for the distilroberta (jhart) model: 61
Tie moods in trainbus for the distilroberta (jhart) model: 69
Tie moods in library for the distilroberta (jhart) model: 31
Tie moods in restaurant for the distilroberta (jhart) model: 115
Tie moods in office for the distilroberta (jhart) model: 121
Tie moods in bar for the distilroberta (jhart) model: 62
Tie moods in mall for the distilroberta (jhart) model: 35


## From the output of the previous cells, there are moods that were tied. These moods were classified as mixed mood

In [15]:
def replace_mixed_mood_multiple_columns(df, columns):
    """
    Replaces tie-moods in list in the specified columns of the DataFrame with 'mixed mood'.

    Parameters:
    df (pandas.DataFrame): The DataFrame to modify.
    columns (list of str): The columns where list-type values should be replaced.

    Returns:
    pandas.DataFrame: The modified DataFrame with list-type values replaced by 'mixed mood' in the specified columns.
    """
    for column in columns:
        df[column] = df[column].apply(lambda x: 'mixed mood' if isinstance(x, list) else x)
    return df


In [16]:
# Replace tied-mood values with 'mixed mood' in the Roberta, Distilroberta Mitch and Distilroberta Mood columns
bedroom_rankMerged_split = replace_mixed_mood_multiple_columns(bedroom_rankMerged_split, ['final_mood_roberta', 'final_mood_mitch', 'final_mood_jhart'])
gym_rankMerged_split = replace_mixed_mood_multiple_columns(gym_rankMerged_split, ['final_mood_roberta', 'final_mood_mitch', 'final_mood_jhart'])
trainbus_rankMerged_split = replace_mixed_mood_multiple_columns(trainbus_rankMerged_split, ['final_mood_roberta', 'final_mood_mitch', 'final_mood_jhart'])
library_rankMerged_split = replace_mixed_mood_multiple_columns(library_rankMerged_split, ['final_mood_roberta', 'final_mood_mitch', 'final_mood_jhart'])
restaurant_rankMerged_split = replace_mixed_mood_multiple_columns(restaurant_rankMerged_split, ['final_mood_roberta', 'final_mood_mitch', 'final_mood_jhart'])
office_rankMerged_split = replace_mixed_mood_multiple_columns(office_rankMerged_split, ['final_mood_roberta', 'final_mood_mitch', 'final_mood_jhart'])
bar_rankMerged_split = replace_mixed_mood_multiple_columns(bar_rankMerged_split, ['final_mood_roberta', 'final_mood_mitch', 'final_mood_jhart'])
mall_rankMerged_split = replace_mixed_mood_multiple_columns(mall_rankMerged_split, ['final_mood_roberta', 'final_mood_mitch', 'final_mood_jhart'])

# Feature Engineering: Participant Ranking and Target Group

## Pre-process participant ranking to binary label. 1 for average ranking >= 3 and 0 for average ranking <3

In [17]:
import pandas as pd

def process_and_concat_categories_with_venue(df, ranking_columns, non_ranking_columns, venue_name="bar"):
    # Only keep columns from non_ranking_columns that exist in the current dataframe
    available_columns = [col for col in non_ranking_columns if col in df.columns]
    
    category_dfs = []
    
    # Process each ranking column separately
    for ranking_col in ranking_columns:
        if ranking_col not in df.columns:
            continue  # Skip if the ranking column doesn't exist in the dataframe
        
        # Create a copy of the relevant data
        category_df = df[available_columns + [ranking_col]].copy()
        
        # Create binary labels based on the ranking
        # 1 for relevant (>=3), 0 for not relevant (<3)
        category_df['label'] = category_df[ranking_col].apply(lambda x: 1 if x >= 3 else 0)
        
        # Drop the ranking column as it's no longer needed
        category_df = category_df.drop(columns=[ranking_col])
        
        # Create a new target group label that includes both the venue and age-gender group
        category_name = ranking_col.replace("_AR", "")  # Strip the _AR part from ranking column
        category_df['target_group'] = f"{venue_name}_{category_name}"
        
        # Append the processed dataframe to the list
        category_dfs.append(category_df)
    
    # Concatenate all the category-specific dataframes into one
    final_df = pd.concat(category_dfs, ignore_index=True)
    
    return final_df


ranking_columns = ['21_25_Female_AR', '21_25_Male_AR', '26_30_Female_AR', '26_30_Male_AR', '31_35_Male_AR', '31_35_Female_AR']
non_ranking_columns = ['S/N', 'track_id', 'track_name', 'artist_name', 'artist_id', 'release_date', 
                       'genre', 'key', 'liveness', 'loudness', 'mode', 'popularity', 'acousticness', 
                       'speechiness', 'tempo', 'valence', 'danceability', 'duration_ms', 'energy', 
                       'lyrics', 'cleaned_lyrics', 'Section_1', 'Section_2', 'Section_3', 'Section_4', 
                       'Section_5', 'Section_6', 'Section_7', 'Section_8', 'Section_9', 'Section_10',
                       'section_moods', 'section_mood_mitch', 'section_mood_jhart', 'final_mood_roberta', 
                       'final_mood_mitch', 'final_mood_jhart']

# Run function on the location dataframes
gym = process_and_concat_categories_with_venue(gym_rankMerged_split, ranking_columns, non_ranking_columns, venue_name="gym")
bedroom = process_and_concat_categories_with_venue(bedroom_rankMerged_split, ranking_columns, non_ranking_columns, venue_name="bedroom")
trainbus = process_and_concat_categories_with_venue(trainbus_rankMerged_split, ranking_columns, non_ranking_columns, venue_name="trainbus")
library = process_and_concat_categories_with_venue(library_rankMerged_split, ranking_columns, non_ranking_columns, venue_name="library")
restaurant = process_and_concat_categories_with_venue(restaurant_rankMerged_split, ranking_columns, non_ranking_columns, venue_name="restaurant")
office = process_and_concat_categories_with_venue(office_rankMerged_split, ranking_columns, non_ranking_columns, venue_name="office")
bar = process_and_concat_categories_with_venue(bar_rankMerged_split, ranking_columns, non_ranking_columns, venue_name="bar")
mall = process_and_concat_categories_with_venue(mall_rankMerged_split, ranking_columns, non_ranking_columns, venue_name="mall")

# Merge all the location data together 

In [19]:
# List of all the dataframes
dfs = [gym, bedroom, trainbus, library, restaurant, office, bar, mall]

# Concatenate all dataframes into one
final_df = pd.concat(dfs, ignore_index=True)

In [20]:
final_df

,S/N,track_id,track_name,artist_name,artist_id,release_date,genre,key,liveness,loudness,...,Section_9,Section_10,section_moods,section_mood_mitch,section_mood_jhart,final_mood_roberta,final_mood_mitch,final_mood_jhart,label,target_group
0,1,52eIcoLUM25zbQupAZYoFh,redrum,21 Savage,1URnnhqYAYcrqrcwql10ft,2024-01-12,"['atl hip hop', 'hip hop', 'rap']",2,0.3750,-8.757,...,NaN,NaN,"['neutral', 'neutral', 'neutral', 'neutral', '...","['neutral', 'neutral', 'joy', 'neutral', 'disg...","['neutral', 'anger', 'disgust', 'neutral', 'an...",neutral,neutral,anger,1,gym_21_25_Female
1,2,4xhsWYTOGcal8zt0J161CU,Lovin On Me,Jack Harlow,2LIk90788K0zvyj2JJVwkJ,2023-11-10,"['deep underground hip hop', 'hip hop', 'kentu...",2,0.0937,-4.911,...,NaN,NaN,"['love', 'neutral', 'love', 'neutral', 'neutra...","['joy', 'neutral', 'joy', 'anger', 'anger', 'j...","['fear', 'disgust', 'fear', 'anger', 'anger', ...",love,joy,fear,1,gym_21_25_Female
2,3,0nlERwr8m4Ks1RYYuTVm0C,Happy Birthday,Busy Signal,4RfTXjK9aiiIKDaKUHpL57,2024-01-24,"['dancehall', 'jamaican dancehall', 'reggae fu...",5,0.1210,-3.665,...,NaN,NaN,"['excitement', 'gratitude', None, None, None, ...","['joy', 'sadness', None, None, None, None, Non...","['joy', 'joy', None, None, None, None, None, N...",mixed mood,mixed mood,joy,1,gym_21_25_Female
3,4,6gBFPUFcJLzWGx4lenP6h2,goosebumps,Travis Scott,0Y5tJX1MQlPlqiwlOH1tJY,2016-09-16,"['rap', 'slap house']",7,0.1490,-3.370,...,NaN,NaN,"['neutral', 'neutral', 'neutral', 'desire', 'n...","['neutral', 'neutral', 'joy', 'fear', 'joy', '...","['sadness', 'neutral', 'sadness', 'fear', 'ang...",neutral,neutral,sadness,1,gym_21_25_Female
4,5,4rXLjWdF2ZZpXCVTfWcshS,fukumean,Gunna,2hlmm7s2ICUX0LVIhVFlZQ,2023-06-16,"['atl hip hop', 'melodic rap', 'rap', 'trap']",1,0.2850,-6.747,...,NaN,NaN,"['amusement', 'neutral', 'optimism', 'neutral'...","['anger', 'anger', 'anger', 'fear', None, None...","['anger', 'anger', 'anger', 'anger', None, Non...",neutral,anger,anger,1,gym_21_25_Female
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25747,239,5YqpHuXpFjDVZ7tY1ClFll,Heartbeats,José González,6xrCU6zdcSTsG2hLrojpmI,03/04/2006,"['chamber pop', 'gothenburg indie', 'indie fol...",3,0.0958,-14.891,...,NaN,NaN,"['neutral', 'disapproval', None, None, None, N...","['joy', 'neutral', None, None, None, None]","['fear', 'sadness', None, None, None, None]",mixed mood,mixed mood,mixed mood,1,mall_31_35_Female
25748,240,1aRvUHgMe9ichgcHAAs12f,Nellie,Dr. Dog,4mLJ3XfOM5FPjSAWdQ2Jk7,01/10/2013,"['chamber pop', 'indie rock', 'modern folk roc...",4,0.0907,-9.936,...,NaN,NaN,"['curiosity', 'desire', None, None, None, None]","['fear', 'neutral', None, None, None, None]","['neutral', 'joy', None, None, None, None]",mixed mood,mixed mood,mixed mood,0,mall_31_35_Female
25749,241,5pD6DyYeglUby2x5CXrqfA,I Am Not Afraid,Charley Crockett,3BJX1nYizKvWpZTY5HOAr4,06/06/2016,"['new americana', 'southern americana']",11,0.1300,-6.539,...,NaN,NaN,"['caring', 'love', None, None, None, None]","['joy', 'fear', None, None, None, None]","['fear', 'fear', None, None, None, None]",mixed mood,mixed mood,fear,1,mall_31_35_Female
25750,242,6f2unaaUhIqTkbHsWMbGcm,Sway,Kacey Musgraves,70kkdajctXSbqSMJbQO424,15/03/2024,"['classic texas country', 'country dawn']",11,0.1020,-11.042,...,NaN,NaN,"['neutral', 'neutral', None, None, None, None]","['fear', 'joy', None, None, None, None]","['fear', 'neutral', None, None, None, None]",neutral,mixed mood,mixed mood,1,mall_31_35_Female


# Feature Engineering: Genre
As stated in the report, the genre collected using spotify artist in the notebook titled "Genre Data Collection", contained multiple genres associated with the arist. I took out the first genre on that list.

In [23]:
import ast

# Function to handle different genre formats and return 'not given' when genre is missing or invalid
def process_genre(genre):
    if isinstance(genre, str):
        # If the string looks like a list, try to parse it
        try:
            genre = ast.literal_eval(genre)
        except (ValueError, SyntaxError):
            # If it's just a string (e.g., 'pop, viral pop'), return the first genre or 'not given'
            return genre.split(", ")[0] if len(genre) > 0 else 'not given'
    # If genre is already a list, return the first element, else return 'not given'
    return genre[0] if isinstance(genre, list) and len(genre) > 0 else 'not given'

# Apply the function to the 'genre' column
final_df['genre'] = final_df['genre'].apply(process_genre)

# Check if 'unknown' replaces NaN
print(final_df['genre'].head())


0                 atl hip hop
1    deep underground hip hop
2                   dancehall
3                         rap
4                 atl hip hop
Name: first_genre, dtype: object


In [24]:
final_df.to_csv('new_dataset.csv', index=False) # Save the completely cleaned dataset

# Data Preparation for Modelling

## Feature Selection and Creating three datasets Roberta, Distilroberta_mitch and Distilroberta for model training, with the aim to evaluate the BERT models used for mood classification. 

In [72]:

#Select the specified columns
selected_columns = ['release_date', 'artist_name', 'liveness', 'loudness', 'mode', 'popularity',
                    'acousticness', 'speechiness', 'tempo', 'valence', 'danceability', 'energy',
                    'final_mood_roberta', 'final_mood_mitch', 'final_mood_jhart', 'first_genre',
                    'target_group', 'label']

# Filter the dataframe to only keep the selected columns
final_df = final_df[selected_columns]
final_df['text_distilroberta_mitch'] = final_df['final_mood_mitch'] + ' ' + final_df['genre']
final_df['text_roberta'] = final_df['final_mood_roberta'] + ' ' + final_df['genre']
final_df['text_distilroberta'] = final_df['final_mood_jhart'] + ' ' + final_df['genre']

Y = filtered_df['label']

X_distilroberta_mitch = final_df.drop(columns=['label', 'final_mood_roberta', 'final_mood_jhart',
                                               'text_roberta', 'text_distilroberta'])

X_distilroberta = final_df.drop(columns=['label', 'final_mood_mitch', 'final_mood_roberta', 
                                         'text_distilroberta_mitch', 'text_roberta'])

X_roberta = final_df.drop(columns=['label', 'final_mood_mitch', 'final_mood_jhart', 
                                   'text_distilroberta_mitch', 'text_distilroberta'])

In [73]:
# Check if it is a balanced dataset:
final_df['label'].value_counts()

label
1    13888
0    11864
Name: count, dtype: int64

In [74]:
# Unique target group (context)
final_df['target_group'].unique()

array(['gym_21_25_Female', 'gym_21_25_Male', 'gym_26_30_Female',
       'gym_26_30_Male', 'gym_31_35_Male', 'gym_31_35_Female',
       'bedroom_21_25_Female', 'bedroom_21_25_Male',
       'bedroom_26_30_Female', 'bedroom_26_30_Male', 'bedroom_31_35_Male',
       'bedroom_31_35_Female', 'trainbus_21_25_Female',
       'trainbus_21_25_Male', 'trainbus_26_30_Female',
       'trainbus_26_30_Male', 'trainbus_31_35_Male',
       'trainbus_31_35_Female', 'library_21_25_Female',
       'library_21_25_Male', 'library_26_30_Female', 'library_26_30_Male',
       'library_31_35_Male', 'library_31_35_Female',
       'restaurant_21_25_Female', 'restaurant_21_25_Male',
       'restaurant_26_30_Female', 'restaurant_26_30_Male',
       'restaurant_31_35_Male', 'restaurant_31_35_Female',
       'office_21_25_Female', 'office_21_25_Male', 'office_26_30_Female',
       'office_26_30_Male', 'office_31_35_Male', 'office_31_35_Female',
       'bar_21_25_Female', 'bar_21_25_Male', 'bar_26_30_Female',
       '

## Training and Testing Split

In [75]:
from sklearn.model_selection import train_test_split

X_train_roberta, 
X_test_roberta, y_train, 
y_test,target_train, target_test, 
train_index, test_index = train_test_split(X_roberta,Y,X_roberta['target_group'],
                                           X_roberta.index,test_size=0.2,
                                           random_state=42,
                                           stratify=X_roberta['target_group'])


# Use the train and test indices obtained from splitting X_roberta to subset the other feature sets

X_train_distilroberta = X_distilroberta.loc[train_index]
X_test_distilroberta = X_distilroberta.loc[test_index]

X_train_distilroberta_mitch = X_distilroberta_mitch.loc[train_index]
X_test_distilroberta_mitch = X_distilroberta_mitch.loc[test_index]

## Save the data for modelling in SupervisedLDA Notebook, MLP Notebook and RandomForest Notebook

In [76]:
# Save feature data as CSV
X_train_roberta.to_csv('X_train_roberta.csv', index=False)
X_test_roberta.to_csv('X_test_roberta.csv', index=False)

X_train_distilroberta.to_csv('X_train_distilroberta.csv', index=False)
X_test_distilroberta.to_csv('X_test_distilroberta.csv', index=False)

X_train_distilroberta_mitch.to_csv('X_train_distilroberta_mitch.csv', index=False)
X_test_distilroberta_mitch.to_csv('X_test_distilroberta_mitch.csv', index=False)

# Save target variables as CSV
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

target_train.to_csv('target_train.csv', index=False)
target_test.to_csv('target_test.csv', index=False)